In [9]:
# ==============================================================================
# SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP
# ==============================================================================
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, coalesce, greatest, round, lead, lag, row_number, desc,
    max as f_max, min as f_min, explode, array
)
from pyspark.sql.window import Window

print("=== SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP ===")

# Configuración inicial
fecha_actual = datetime.now()
ano_actual = int(fecha_actual.strftime("%Y"))
print(f"Año de ejecución: {ano_actual}")

# Cargar tabla base del Lakehouse
df_master_basico_raw = spark.sql("""
    SELECT * 
    FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.continental_master_basic_daily
""")

print(f"Registros cargados (raw): {df_master_basico_raw.count():,}")

# Deduplicación completa por clave primaria
w_dedup = Window.partitionBy("PAIS", "Familia", "Ano", "ORDENM", "PWOH").orderBy("Mes")
df_master_basico = (
    df_master_basico_raw
    .withColumn("rn", row_number().over(w_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f"Registros tras deduplicación: {df_master_basico.count():,}")

# Sample inicial CR-LAVA
print("\nSAMPLE INICIAL - CR-LAVA 2026:")
(df_master_basico
 .filter((col("PAIS") == "CR") & (col("Familia") == "LAVA") & (col("Ano") == 2026))
 .select("Attribute", "ORDENM", 
         F.round("Inv_Inicial", 0).alias("Inv_Inic"),
         F.round("Llegadas", 0).alias("Llegadas"),
         F.round("Fcst", 0).alias("S&OP"),
         F.round("Facturacion", 0).alias("Facturado"),
         F.round("Inv_Final", 0).alias("Inv_Final"))
 .orderBy("ORDENM").show(12, truncate=False))

print("[COMPLETADO] Bootstrap y carga de datos")

StatementMeta(, 9da9541c-86ad-424d-a3a8-e1790cfa809a, 11, Finished, Available, Finished)

=== SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP ===
Año de ejecución: 2025
Registros cargados (raw): 4,608
Registros tras deduplicación: 4,608

SAMPLE INICIAL - CR-LAVA 2026:
+---------+------+--------+--------+------+---------+---------+
|Attribute|ORDENM|Inv_Inic|Llegadas|S&OP  |Facturado|Inv_Final|
+---------+------+--------+--------+------+---------+---------+
|ENE      |1     |0.0     |1329.0  |2298.0|0.0      |0.0      |
|FEB      |2     |0.0     |1632.0  |1700.0|0.0      |0.0      |
|MAR      |3     |0.0     |0.0     |2896.0|0.0      |0.0      |
|ABR      |4     |0.0     |0.0     |2296.0|0.0      |0.0      |
|MAY      |5     |0.0     |0.0     |2680.0|0.0      |0.0      |
|JUN      |6     |0.0     |0.0     |1644.0|0.0      |0.0      |
|JUL      |7     |0.0     |0.0     |2767.0|0.0      |0.0      |
|AGO      |8     |0.0     |0.0     |2314.0|0.0      |0.0      |
|SEP      |9     |0.0     |0.0     |1659.0|0.0      |0.0      |
|OCT      |10    |0.0     |0.0     |2425.0|0.0      |0.0      |

In [2]:
# ==============================================================================
# SNIPPET 2: DETECCIÓN DEL MES ANCLA
# ==============================================================================
print("=== SNIPPET 2: DETECCIÓN DEL MES ANCLA ===")

print("LÓGICA: Mes ancla = ÚLTIMO mes con Facturación > 0")
print("PROPÓSITO: Define punto de corte entre datos reales y proyecciones")

# Ventana por año para detectar mes ancla
w_año = Window.partitionBy("PAIS", "Familia", "Ano")

df_con_ancla = (
    df_master_basico
    # Detectar candidatos: meses con facturación > 0
    .withColumn("mes_candidato_ancla", 
                when(coalesce(col("Facturacion"), lit(0.0)) > 0.0, col("ORDENM")))
    
    # Mes ancla = último (máximo) mes con facturación
    .withColumn("mes_ancla", f_max("mes_candidato_ancla").over(w_año))
    
    # Fallback: si no hay facturación, usar mes 1
    .withColumn("mes_ancla", coalesce(col("mes_ancla"), lit(1)))
    
    .drop("mes_candidato_ancla")
    
    # Flags de posición relativa al ancla
    .withColumn("es_mes_ancla", col("ORDENM") == col("mes_ancla"))
    .withColumn("antes_del_ancla", col("ORDENM") < col("mes_ancla"))
    .withColumn("despues_del_ancla", col("ORDENM") > col("mes_ancla"))
)

# Verificación de anclas detectadas
print("\nVerificación anclas detectadas:")
(df_con_ancla.filter(col("es_mes_ancla") == True)
 .select("PAIS", "Familia", "Ano", "mes_ancla", "Attribute", 
         F.round("Facturacion", 0).alias("Facturacion"))
 .distinct().orderBy("PAIS", "Familia", "Ano").show(10, truncate=False))

# Sample CR-LAVA con ancla
print("\nSAMPLE CR-LAVA - Detección ancla:")
(df_con_ancla
 .filter((col("PAIS") == "CR") & (col("Familia") == "LAVA") & (col("Ano") == 2025))
 .select("Attribute", "ORDENM", "mes_ancla", "es_mes_ancla", "antes_del_ancla", "despues_del_ancla",
         F.round("Facturacion", 0).alias("Facturado"))
 .orderBy("ORDENM").show(12, truncate=False))

print("[COMPLETADO] Detección del mes ancla")

StatementMeta(, 3117fe65-057e-41cd-b707-9e776779243e, 4, Finished, Available, Finished)

=== SNIPPET 2: DETECCIÓN DEL MES ANCLA ===
LÓGICA: Mes ancla = ÚLTIMO mes con Facturación > 0
PROPÓSITO: Define punto de corte entre datos reales y proyecciones

Verificación anclas detectadas:
+----+-------+----+---------+---------+-----------+
|PAIS|Familia|Ano |mes_ancla|Attribute|Facturacion|
+----+-------+----+---------+---------+-----------+
|AR  |COCC   |2025|11       |NOV      |417.0      |
|AR  |COCC   |2026|1        |ENE      |0.0        |
|AR  |COCC   |2027|1        |ENE      |0.0        |
|AR  |COCC   |2028|1        |ENE      |0.0        |
|AR  |COCC   |2029|1        |ENE      |0.0        |
|AR  |COCC   |2030|1        |ENE      |0.0        |
|AR  |GLOB   |2025|11       |NOV      |729.0      |
|AR  |GLOB   |2026|1        |ENE      |0.0        |
|AR  |GLOB   |2027|1        |ENE      |0.0        |
|AR  |GLOB   |2028|1        |ENE      |0.0        |
+----+-------+----+---------+---------+-----------+
only showing top 10 rows


SAMPLE CR-LAVA - Detección ancla:
+---------+------

In [3]:
# ==============================================================================
# SNIPPET 3: CÁLCULO LLEGADAS Y RECÁLCULO INVENTARIOS - LÓGICA FINAL
# ==============================================================================
print("=== SNIPPET 3: LLEGADAS E INVENTARIOS - LÓGICA CORRECTA ===")

print("LÓGICAS:")
print("1. ANTES del ancla: Calcular llegadas donde están en 0, mantener Inv_Final del Daily")
print("2. MES ANCLA: Mantener TODO del Daily (Inv_Final YA está correcto)")
print("3. DESPUÉS del ancla: CALCULAR Inv_Final usando S&OP")

# Ventana multi-año
w_continuo = Window.partitionBy("PAIS", "Familia").orderBy("Ano", "ORDENM")

# Detectar último mes con S&OP
df_con_limites = (
    df_con_ancla
    .withColumn("mes_candidato_fcst", 
                when(coalesce(col("Fcst"), lit(0.0)) > 0.0, col("ORDENM")))
    .withColumn("ultimo_mes_fcst", f_max("mes_candidato_fcst").over(w_año))
    .withColumn("ultimo_mes_fcst", coalesce(col("ultimo_mes_fcst"), lit(12)))
    .drop("mes_candidato_fcst")
)

# Aplicar cálculos iterativos
df_calculado = df_con_limites

for iteracion in range(1, 6):
    print(f"  Iteración {iteracion}...")
    
    # PASO A: Continuidad Inv_Inicial[t] = Inv_Final[t-1]
    df_calculado = (
        df_calculado
        .withColumn("Inv_Final_anterior", lag("Inv_Final").over(w_continuo))
        .withColumn(
            "Inv_Inicial_nuevo",
            when(col("ORDENM") == 1,
                 coalesce(col("Inv_Final_anterior"), col("Inv_Inicial")))
            .otherwise(coalesce(col("Inv_Final_anterior"), col("Inv_Inicial")))
        )
        .drop("Inv_Final_anterior")
    )
    
    # PASO B: Calcular llegadas ANTES del ancla donde están en 0
    df_calculado = df_calculado.withColumn(
        "Llegadas_nuevo",
        when(
            col("antes_del_ancla") & (coalesce(col("Llegadas"), lit(0.0)) == 0.0),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Final"), lit(0.0)) -
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) +
                coalesce(col("Facturacion"), lit(0.0))
            )
        ).otherwise(coalesce(col("Llegadas"), lit(0.0)))
    )
    
    # PASO C: Recalcular Inv_Final SOLO DESPUÉS del ancla
    df_calculado = df_calculado.withColumn(
        "Inv_Final_nuevo",
        when(
            # DESPUÉS del ancla: CALCULAR con S&OP
            col("despues_del_ancla") & 
            (col("ORDENM") <= col("ultimo_mes_fcst")) &
            (coalesce(col("Fcst"), lit(0.0)) > 0.0),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) +
                coalesce(col("Llegadas_nuevo"), lit(0.0)) -
                coalesce(col("Fcst"), lit(0.0))
            )
        ).otherwise(
            # ANTES del ancla Y MES ANCLA: Mantener del Daily
            coalesce(col("Inv_Final"), lit(0.0))
        )
    )
    
    # PASO D: Actualizar columnas
    df_calculado = (
        df_calculado
        .drop("Inv_Inicial", "Llegadas", "Inv_Final")
        .withColumnRenamed("Inv_Inicial_nuevo", "Inv_Inicial")
        .withColumnRenamed("Llegadas_nuevo", "Llegadas")
        .withColumnRenamed("Inv_Final_nuevo", "Inv_Final")
    )

df_master_corregido = (
    df_calculado
    .select(
        col("Ano"), col("Mes"), col("PAIS"), col("Familia"), col("Attribute"),
        col("ORDENM"), col("Inv_Inicial"), col("Llegadas"), col("Fcst"), 
        col("Facturacion"), col("Inv_Final"),
        col("mes_ancla"), col("ultimo_mes_fcst"), col("es_mes_ancla"), 
        col("antes_del_ancla"), col("despues_del_ancla")
    )
).cache()

print(f"Tabla base corregida: {df_master_corregido.count():,}")
print("[COMPLETADO] Cálculo correcto de inventarios")


StatementMeta(, 3117fe65-057e-41cd-b707-9e776779243e, 5, Finished, Available, Finished)

=== SNIPPET 3: LLEGADAS E INVENTARIOS - LÓGICA CORRECTA ===
LÓGICAS:
1. ANTES del ancla: Calcular llegadas donde están en 0, mantener Inv_Final del Daily
2. MES ANCLA: Mantener TODO del Daily (Inv_Final YA está correcto)
3. DESPUÉS del ancla: CALCULAR Inv_Final usando S&OP
  Iteración 1...
  Iteración 2...
  Iteración 3...
  Iteración 4...
  Iteración 5...
Tabla base corregida: 4,608
[COMPLETADO] Cálculo correcto de inventarios


In [4]:
# ==============================================================================
# SNIPPET 4: LOOP PWOH 1-10
# ==============================================================================
print("=== SNIPPET 4: LOOP PWOH 1-10 ===")

print("LÓGICA:")
print("1. Crear 10 versiones (PWOH 1-10) de cada registro")
print("2. Aplicar fórmula PWOH solo DESPUÉS del mes ancla")
print("3. Recalcular Inv_Final con las nuevas llegadas PWOH")

# Cross join con PWOH 1-10
pwoh_array = [lit(i) for i in range(1, 11)]
df_pwoh_base = (
    df_master_corregido
    .withColumn("pwoh_array", array(*pwoh_array))
    .withColumn("PWOH", explode(col("pwoh_array")))
    .drop("pwoh_array")
)

# Ventanas para forecasts
w_intra = Window.partitionBy("PAIS", "Familia", "Ano", "PWOH").orderBy("ORDENM")
w_multi = Window.partitionBy("PAIS", "Familia", "PWOH").orderBy("Ano", "ORDENM")

# Aplicar fórmula PWOH
df_pwoh_final = (
    df_pwoh_base
    .withColumn("fcst_sig_intra", lead("Fcst", 1).over(w_intra))
    .withColumn("fcst_sig_multi", lead("Fcst", 1).over(w_multi))
    
    # Condición: Solo aplicar PWOH después del ancla donde llegadas = 0 hasta último mes con forecast
    .withColumn("aplica_pwoh", 
                col("despues_del_ancla") & 
                (col("ORDENM") <= col("ultimo_mes_fcst")) &
                (coalesce(col("Llegadas"), lit(0.0)) == 0.0))  # Solo donde llegadas = 0
    
    # Fórmula PWOH para llegadas
    .withColumn(
        "Llegadas_pwoh",
        when(
            col("aplica_pwoh"),
            when(
                col("ORDENM") == 12,  # Diciembre especial
                greatest(lit(0.0),
                    ((coalesce(col("Fcst"), lit(0.0)) + 
                      coalesce(col("fcst_sig_multi"), col("Fcst"), lit(0.0))) / 4.0 * col("PWOH")) +
                    coalesce(col("Fcst"), lit(0.0)) - coalesce(col("Inv_Inicial"), lit(0.0))
                )
            ).otherwise(  # Meses normales
                greatest(lit(0.0),
                    (coalesce(col("fcst_sig_intra"), lit(0.0)) / 4.0 * col("PWOH")) +
                    coalesce(col("Fcst"), lit(0.0)) - coalesce(col("Inv_Inicial"), lit(0.0))
                )
            )
        ).otherwise(coalesce(col("Llegadas"), lit(0.0)))  # Mantener originales
    )
    
    # Recalcular Inv_Final con llegadas PWOH
    .withColumn(
        "Inv_Final_pwoh",
        when(
            col("aplica_pwoh"),
            greatest(lit(0.0),
                coalesce(col("Inv_Inicial"), lit(0.0)) +
                coalesce(col("Llegadas_pwoh"), lit(0.0)) -
                coalesce(col("Fcst"), lit(0.0))
            )
        ).otherwise(coalesce(col("Inv_Final"), lit(0.0)))
    )
    
    .select("Ano", "Mes", "PAIS", "Familia", "Attribute", "ORDENM", "PWOH",
            col("Inv_Inicial").alias("Inv_Inic"),
            col("Llegadas_pwoh").alias("Llegadas"),
            "Fcst", "Facturacion",
            col("Inv_Final_pwoh").alias("Inv_Final"),
            "mes_ancla", "aplica_pwoh")
)

df_pwoh_final.cache()
print(f"PWOH completado: {df_pwoh_final.count():,} registros")

# Sample CR-LAVA PWOH
print("\nSAMPLE CR-LAVA - PWOH 1,5,10:")
(df_pwoh_final
 .filter((col("PAIS") == "CR") & (col("Familia") == "LAVA") & (col("Ano") == 2025) & col("PWOH").isin(1,5,10))
 .select("PWOH", "Attribute", "ORDENM", "aplica_pwoh",
         F.round("Inv_Inic", 0).alias("Inv_Inic"),
         F.round("Llegadas", 0).alias("Llegadas"),
         F.round("Fcst", 0).alias("S&OP"),
         F.round("Inv_Final", 0).alias("Inv_Final"))
 .orderBy("PWOH", "ORDENM").show(36, truncate=False))

print("[COMPLETADO] Loop PWOH 1-10")

StatementMeta(, 3117fe65-057e-41cd-b707-9e776779243e, 6, Finished, Available, Finished)

=== SNIPPET 4: LOOP PWOH 1-10 ===
LÓGICA:
1. Crear 10 versiones (PWOH 1-10) de cada registro
2. Aplicar fórmula PWOH solo DESPUÉS del mes ancla
3. Recalcular Inv_Final con las nuevas llegadas PWOH
PWOH completado: 46,080 registros

SAMPLE CR-LAVA - PWOH 1,5,10:
+----+---------+------+-----------+--------+--------+------+---------+
|PWOH|Attribute|ORDENM|aplica_pwoh|Inv_Inic|Llegadas|S&OP  |Inv_Final|
+----+---------+------+-----------+--------+--------+------+---------+
|1   |ENE      |1     |false      |3656.0  |4711.0  |1539.0|7049.0   |
|1   |FEB      |2     |false      |7049.0  |2759.0  |1539.0|9011.0   |
|1   |MAR      |3     |false      |9011.0  |65.0    |1319.0|8191.0   |
|1   |ABR      |4     |false      |8191.0  |0.0     |1405.0|5782.0   |
|1   |MAY      |5     |false      |5782.0  |77.0    |1696.0|4139.0   |
|1   |JUN      |6     |false      |4139.0  |681.0   |2706.0|4174.0   |
|1   |JUL      |7     |false      |4174.0  |368.0   |1416.0|2501.0   |
|1   |AGO      |8     |false

In [5]:
# ==============================================================================
# SNIPPET 5: CÁLCULO DE TTOS
# ==============================================================================
print("=== SNIPPET 5: CÁLCULO DE TTOS ===")

print("LÓGICA:")
print("TTOS = Llegadas[t+1] + (Llegadas[t+2] / 2)")
print("Casos especiales: NOV y DIC usan llegadas multi-año")

# Ventanas para llegadas futuras
w_intra_ttos = Window.partitionBy("PAIS", "Familia", "Ano", "PWOH").orderBy("ORDENM")
w_multi_ttos = Window.partitionBy("PAIS", "Familia", "PWOH").orderBy("Ano", "ORDENM")

df_ttos = (
    df_pwoh_final
    .withColumn("lleg_m1_intra", lead("Llegadas", 1).over(w_intra_ttos))
    .withColumn("lleg_m2_intra", lead("Llegadas", 2).over(w_intra_ttos))
    .withColumn("lleg_m1_multi", lead("Llegadas", 1).over(w_multi_ttos))
    .withColumn("lleg_m2_multi", lead("Llegadas", 2).over(w_multi_ttos))
    
    # Cálculo TTOS con casos especiales
    .withColumn(
        "TTOS",
        when(
            col("ORDENM") == 11,  # NOV
            coalesce(col("lleg_m1_intra"), lit(0.0)) + 
            (coalesce(col("lleg_m2_multi"), col("lleg_m1_intra"), lit(0.0)) / 2.0)
        ).when(
            col("ORDENM") == 12,  # DIC
            coalesce(col("lleg_m1_multi"), col("lleg_m1_intra"), lit(0.0)) +
            (coalesce(col("lleg_m2_multi"), col("lleg_m2_intra"), col("lleg_m1_multi"), lit(0.0)) / 2.0)
        ).otherwise(  # Meses normales
            coalesce(col("lleg_m1_intra"), lit(0.0)) + 
            (coalesce(col("lleg_m2_intra"), lit(0.0)) / 2.0)
        )
    )
    .withColumn("UND_TTOS", col("Inv_Final") + col("TTOS"))
    
    .select("Ano", "Mes", "PAIS", "Familia", "Attribute", "ORDENM", "PWOH",
            "Inv_Inic", "Llegadas", "Fcst", "Facturacion", "Inv_Final", 
            "TTOS", "UND_TTOS")
)

df_ttos.cache()
print(f"TTOS completado: {df_ttos.count():,} registros")

# Sample CR-LAVA TTOS
print("\nSAMPLE CR-LAVA - TTOS PWOH 1:")
(df_ttos
 .filter((col("PAIS") == "CR") & (col("Familia") == "LAVA") & (col("Ano") == 2025) & (col("PWOH") == 1))
 .select("Attribute", "ORDENM",
         F.round("Inv_Final", 0).alias("Inv_Final"),
         F.round("TTOS", 0).alias("TTOS"),
         F.round("UND_TTOS", 0).alias("UND_TTOS"))
 .orderBy("ORDENM").show(12, truncate=False))

print("[COMPLETADO] Cálculo de TTOS")

StatementMeta(, 3117fe65-057e-41cd-b707-9e776779243e, 7, Finished, Available, Finished)

=== SNIPPET 5: CÁLCULO DE TTOS ===
LÓGICA:
TTOS = Llegadas[t+1] + (Llegadas[t+2] / 2)
Casos especiales: NOV y DIC usan llegadas multi-año
TTOS completado: 46,080 registros

SAMPLE CR-LAVA - TTOS PWOH 1:
+---------+------+---------+------+--------+
|Attribute|ORDENM|Inv_Final|TTOS  |UND_TTOS|
+---------+------+---------+------+--------+
|ENE      |1     |7049.0   |2792.0|9841.0  |
|FEB      |2     |9011.0   |65.0  |9076.0  |
|MAR      |3     |8191.0   |39.0  |8230.0  |
|ABR      |4     |5782.0   |418.0 |6200.0  |
|MAY      |5     |4139.0   |865.0 |5004.0  |
|JUN      |6     |4174.0   |1749.0|5923.0  |
|JUL      |7     |2501.0   |2994.0|5495.0  |
|AGO      |8     |3495.0   |1316.0|4811.0  |
|SEP      |9     |2916.0   |2085.0|5001.0  |
|OCT      |10    |2943.0   |2373.0|5316.0  |
|NOV      |11    |3407.0   |3874.0|7281.0  |
|DIC      |12    |3143.0   |2145.0|5288.0  |
+---------+------+---------+------+--------+

[COMPLETADO] Cálculo de TTOS


In [6]:
# ==============================================================================
# SNIPPET 6: CÁLCULO DE WOH Y WOH+TTOS
# ==============================================================================
print("=== SNIPPET 6: CÁLCULO DE WOH Y WOH+TTOS ===")

print("LÓGICA:")
print("WOH = Inv_Final / (Forecast_Promedio_Semanal)")
print("Denominador: 8 semanas si mes <= último_facturado, 4 semanas si después")

# Último mes con facturación
df_ultimo_fact = (
    df_ttos.filter(col("Facturacion") > 0)
    .groupBy("PAIS", "Familia", "Ano", "PWOH")
    .agg(f_max("ORDENM").alias("ultimo_fact"))
)

# Fallbacks enero/febrero 2025
df_fallbacks = (
    df_ttos.filter((col("Ano") == 2025) & col("ORDENM").isin(1, 2))
    .groupBy("PAIS", "Familia", "PWOH")
    .pivot("ORDENM", [1, 2]).agg(F.first("Fcst"))
    .withColumnRenamed("1", "fcst_ene_2025").withColumnRenamed("2", "fcst_feb_2025")
)

# Ventanas para forecasts futuros
w_intra_woh = Window.partitionBy("PAIS", "Familia", "Ano", "PWOH").orderBy("ORDENM")

# Cálculo WOH
df_woh = (
    df_ttos
    .join(df_ultimo_fact, ["PAIS", "Familia", "Ano", "PWOH"], "left")
    .join(df_fallbacks, ["PAIS", "Familia", "PWOH"], "left")
    .withColumn("ultimo_fact", coalesce(col("ultimo_fact"), lit(0)))
    .fillna(0, ["fcst_ene_2025", "fcst_feb_2025"])
    .withColumn("fcst_m1", lead("Fcst", 1).over(w_intra_woh))
    .withColumn("fcst_m2", lead("Fcst", 2).over(w_intra_woh))
    
    # Denominador inteligente
    .withColumn(
        "denominador",
        when(
            col("ORDENM") <= col("ultimo_fact"),  # 8 semanas
            when(col("ORDENM") == 11,  # NOV
                (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_ene_2025"), lit(0.0))) / 8.0
            ).when(col("ORDENM") == 12,  # DIC
                (coalesce(col("fcst_ene_2025"), lit(0.0)) + coalesce(col("fcst_feb_2025"), lit(0.0))) / 8.0
            ).otherwise(
                (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0
            )
        ).otherwise(  # 4 semanas
            when(col("ORDENM") == 12,  # DIC
                coalesce(col("fcst_ene_2025"), lit(0.0)) / 4.0
            ).otherwise(
                coalesce(col("fcst_m1"), lit(0.0)) / 4.0
            )
        )
    )
    
    # WOH como enteros
    .withColumn(
        "WOH",
        when((col("Inv_Final") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("Inv_Final") / col("denominador"), 0).cast("int"))
    )
    .withColumn(
        "WOH_TTOS", 
        when((col("UND_TTOS") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("UND_TTOS") / col("denominador"), 0).cast("int"))
    )
    
    .select("Ano", "Mes", "PAIS", "Familia", "Attribute", "ORDENM", "PWOH",
            "Inv_Inic", "Llegadas", "Fcst", "Facturacion", "Inv_Final", 
            "TTOS", "UND_TTOS", "WOH", "WOH_TTOS")
)

df_woh.cache()
print(f"WOH completado: {df_woh.count():,} registros")

# Sample CR-LAVA WOH
print("\nSAMPLE CR-LAVA - WOH PWOH 1:")
(df_woh
 .filter((col("PAIS") == "CR") & (col("Familia") == "LAVA") & (col("Ano") == 2025) & (col("PWOH") == 1))
 .select("Attribute", "ORDENM",
         F.round("Inv_Final", 0).alias("Inv_Final"),
         "WOH", "WOH_TTOS")
 .orderBy("ORDENM").show(12, truncate=False))

print("[COMPLETADO] Cálculo de WOH y WOH+TTOS")
# 

StatementMeta(, 3117fe65-057e-41cd-b707-9e776779243e, 8, Finished, Available, Finished)

=== SNIPPET 6: CÁLCULO DE WOH Y WOH+TTOS ===
LÓGICA:
WOH = Inv_Final / (Forecast_Promedio_Semanal)
Denominador: 8 semanas si mes <= último_facturado, 4 semanas si después
WOH completado: 46,080 registros

SAMPLE CR-LAVA - WOH PWOH 1:
+---------+------+---------+---+--------+
|Attribute|ORDENM|Inv_Final|WOH|WOH_TTOS|
+---------+------+---------+---+--------+
|ENE      |1     |7049.0   |20 |28      |
|FEB      |2     |9011.0   |26 |27      |
|MAR      |3     |8191.0   |21 |21      |
|ABR      |4     |5782.0   |11 |11      |
|MAY      |5     |4139.0   |8  |10      |
|JUN      |6     |4174.0   |9  |13      |
|JUL      |7     |2501.0   |5  |12      |
|AGO      |8     |3495.0   |8  |11      |
|SEP      |9     |2916.0   |7  |11      |
|OCT      |10    |2943.0   |4  |8       |
|NOV      |11    |3407.0   |5  |12      |
|DIC      |12    |3143.0   |8  |14      |
+---------+------+---------+---+--------+

[COMPLETADO] Cálculo de WOH y WOH+TTOS


In [7]:
# ==============================================================================
# SNIPPET 7: MONETARIOS, BUDGET Y GUARDADO FINAL
# ==============================================================================
print("=== SNIPPET 7: MONETARIOS, BUDGET Y GUARDADO FINAL ===")

# Cargar archivos budget con manejo de errores
try:
    df_bgt_pp = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_PP.csv")
        .select(col("Pais").alias("PAIS"), col("Familia"), col("PP").cast("double").alias("BGT_PP"))
    )
    print("BGT_PP cargado exitosamente")
except:
    print("BGT_PP no encontrado, usando defaults")
    df_bgt_pp = spark.createDataFrame([
        ("CR", "LAVA", 164.5), ("CO", "LAVA", 100.0), ("MX", "SAGA", 120.0)
    ], ["PAIS", "Familia", "BGT_PP"])

try:
    df_bgt_cont = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_CONT.csv")
        .select("PAIS", "Familia", "Attribute", col("ORDENM").cast("int"), 
                col("BGT_QTY").cast("double"), col("BGT_$").cast("double").alias("BGT_USD"))
    )
    print("BGT_CONT cargado exitosamente")
except:
    print("BGT_CONT no encontrado, usando defaults")
    df_bgt_cont = spark.createDataFrame([
        ("CR", "LAVA", "ENE", 1, 2550.0, 100000.0),
        ("CR", "LAVA", "FEB", 2, 2219.0, 95000.0),
        ("CO", "LAVA", "ENE", 1, 1000.0, 80000.0),
        ("MX", "SAGA", "ENE", 1, 1500.0, 120000.0)
    ], ["PAIS", "Familia", "Attribute", "ORDENM", "BGT_QTY", "BGT_USD"])

# Calcular monetarios
df_con_monetarios = (
    df_woh
    .join(df_bgt_pp, ["PAIS", "Familia"], "left")
    .join(df_bgt_cont, ["PAIS", "Familia", "Attribute", "ORDENM"], "left")
    .fillna(0, ["BGT_PP", "BGT_QTY", "BGT_USD"])
    
    # Columnas monetarias
    .withColumn("Inv_Final_USD", (col("Inv_Final") * col("BGT_PP")) / 1_000_000)
    .withColumn("TTOS_USD", (col("TTOS") * col("BGT_PP")) / 1_000_000)
    .withColumn("USD_OH_TTOS", col("Inv_Final_USD") + col("TTOS_USD"))
    
    # Diferencias vs budget
    .withColumn("Dif_Real_vs_BGT_QTY", col("UND_TTOS") - col("BGT_QTY"))
    .withColumn("Dif_Real_vs_BGT_USD", col("USD_OH_TTOS") - col("BGT_USD"))
)

print(f"Monetarios agregados: {df_con_monetarios.count():,} registros")

# Tabla final para Power BI
df_tabla_final = (
    df_con_monetarios
    .select(
        col("Ano").cast("int"),
        col("Mes"),
        when(col("PAIS").isin("CO", "EC", "PE", "AR", "CL"), "Andino")
        .when(col("PAIS").isin("CR", "GT", "HN", "SV", "MX"), "CEAM")
        .when(col("PAIS").isin("CA"), "NorteAmerica")
        .otherwise("Otros").alias("Region"),
        col("PAIS"),
        col("Familia"),
        col("Attribute"),
        col("Inv_Inic").cast("double"),
        col("Llegadas").cast("double"),
        col("Fcst").alias("S&PO").cast("double"),
        col("Facturacion").alias("Facturado").cast("double"),
        col("Inv_Final").cast("double"),
        F.round(col("Inv_Final_USD"), 4).alias("Inv_Final_$"),
        col("WOH").alias("WHO").cast("int"),
        col("TTOS").cast("double"),
        F.round(col("TTOS_USD"), 4).alias("TTOS_$"),
        F.round(col("USD_OH_TTOS"), 4).alias("USD_OH_TTOS"),
        col("UND_TTOS").alias("UND_TTO").cast("double"),
        col("WOH_TTOS").alias("WOH_TTO").cast("int"),
        col("BGT_QTY").cast("double"),
        col("BGT_USD").alias("BGT_$").cast("double"),
        col("Dif_Real_vs_BGT_QTY").cast("double"),
        col("Dif_Real_vs_BGT_USD").alias("Dif_Real_vs_BGT_$").cast("double"),
        col("PWOH").cast("double"),
        col("ORDENM").cast("double"),
        lit("Cubos").alias("Calculo")  # COLUMNA IDENTIFICADORA
    )
)

print(f"Tabla final estructurada: {df_tabla_final.count():,} registros")

# ==============================================================================
# ELIMINACIÓN DE DUPLICADOS Y OPTIMIZACIÓN
# ==============================================================================
print("\nEliminando duplicados por clave primaria...")

# Conteo antes de eliminación
count_antes = df_tabla_final.count()

# Eliminar duplicados por clave primaria completa
df_tabla_sin_duplicados = df_tabla_final.dropDuplicates([
    "PAIS", "Familia", "Ano", "ORDENM", "PWOH"
])

count_despues = df_tabla_sin_duplicados.count()
duplicados_eliminados = count_antes - count_despues

print(f"Registros antes: {count_antes:,}")
print(f"Registros después: {count_despues:,}")
print(f"Duplicados eliminados: {duplicados_eliminados:,}")

# Optimización final con particionamiento
df_tabla_final_opt = df_tabla_sin_duplicados.repartition(200, "PAIS", "Familia", "PWOH")

# ==============================================================================
# GUARDADO DE TABLA FINAL
# ==============================================================================
print("\nGuardando tabla final en Lakehouse...")

spark.sql("USE LH_INTELLIGENT_VISIBILITY_CARATULAS")

# Eliminar tabla anterior si existe
try:
    spark.sql("DROP TABLE IF EXISTS tbl_caratula_cdv_clean")
    print("Tabla anterior eliminada")
except Exception as e:
    print(f"No se pudo eliminar tabla anterior: {e}")

# Guardar nueva tabla
df_tabla_final_opt.write.format("delta").mode("overwrite").saveAsTable("tbl_caratula_cdv_clean")

print("TABLA FINAL GUARDADA:")
print("- Nombre: tbl_caratula_cdv_clean")
print("- Esquema: LH_INTELLIGENT_VISIBILITY_CARATULAS")
print(f"- Registros finales: {count_despues:,}")
print(f"- Particiones: 200")

# ==============================================================================
# VERIFICACIÓN FINAL
# ==============================================================================
print("\nVerificación final con sample CR-LAVA:")

# Sample final CR-LAVA PWOH 1
sample_final = spark.sql("""
SELECT Attribute, ORDENM, Inv_Inic, Llegadas, `S&PO`, Facturado, 
       Inv_Final, WHO, TTOS, UND_TTO, `Inv_Final_$`, `TTOS_$`, USD_OH_TTOS
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CR' AND Familia = 'LAVA' AND Ano = 2025 AND PWOH = 1
ORDER BY ORDENM
""")

print("\nSAMPLE FINAL - CR-LAVA PWOH 1:")
sample_final.show(12, truncate=False)

# Verificación sample PWOH 5
print("\nSAMPLE FINAL - CR-LAVA PWOH 5:")
spark.sql("""
SELECT Attribute, ORDENM, Inv_Inic, Llegadas, `S&PO`, Facturado, 
       Inv_Final, WHO, TTOS, UND_TTO
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CR' AND Familia = 'LAVA' AND Ano = 2025 AND PWOH = 5
ORDER BY ORDENM
""").show(12, truncate=False)

# Estadísticas generales de la tabla
print("\nEstadísticas generales:")
spark.sql("""
SELECT 
    COUNT(*) as total_registros,
    COUNT(DISTINCT PAIS) as paises,
    COUNT(DISTINCT Familia) as familias,
    COUNT(DISTINCT Ano) as anos,
    COUNT(DISTINCT PWOH) as pwoh_values
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
""").show()

# Verificación de integridad
print("\nVerificación de integridad:")
spark.sql("""
SELECT 
    PAIS,
    Familia,
    Ano,
    COUNT(*) as registros_por_ano,
    COUNT(DISTINCT ORDENM) as meses_distintos,
    COUNT(DISTINCT PWOH) as pwoh_distintos
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CR' AND Familia = 'LAVA'
GROUP BY PAIS, Familia, Ano
ORDER BY Ano
""").show()

print("\n" + "="*80)
print("PROCESO COMPLETADO - TABLA LISTA PARA POWER BI")
print("="*80)
print("RESUMEN DE CORRECCIONES APLICADAS:")
print("✓ Mes ancla detectado por último mes con facturación")
print("✓ Llegadas calculadas antes del ancla donde están en 0")
print("✓ Llegadas del mes ancla mantenidas sin modificar")
print("✓ Inv_Final en mes ancla calculado con facturación")
print("✓ Inv_Final después del ancla calculado con S&OP")
print("✓ Loop PWOH 1-10 aplicado solo después del ancla donde llegadas = 0")
print("✓ Continuidad multi-año aplicada correctamente")
print("✓ TTOS, WOH y monetarios calculados")
print("✓ Duplicados eliminados por clave primaria")
print("✓ Tabla final optimizada y guardada")
print("="*80)

print("[COMPLETADO] Monetarios, budget y guardado final")

StatementMeta(, 3117fe65-057e-41cd-b707-9e776779243e, 9, Finished, Available, Finished)

=== SNIPPET 7: MONETARIOS, BUDGET Y GUARDADO FINAL ===
BGT_PP cargado exitosamente
BGT_CONT cargado exitosamente
Monetarios agregados: 69,120 registros
Tabla final estructurada: 69,120 registros

Eliminando duplicados por clave primaria...
Registros antes: 69,120
Registros después: 46,080
Duplicados eliminados: 23,040

Guardando tabla final en Lakehouse...
Tabla anterior eliminada
TABLA FINAL GUARDADA:
- Nombre: tbl_caratula_cdv_clean
- Esquema: LH_INTELLIGENT_VISIBILITY_CARATULAS
- Registros finales: 46,080
- Particiones: 200

Verificación final con sample CR-LAVA:

SAMPLE FINAL - CR-LAVA PWOH 1:
+---------+------+--------+--------+------+---------+---------+---+------+-------+-----------+------+-----------+
|Attribute|ORDENM|Inv_Inic|Llegadas|S&PO  |Facturado|Inv_Final|WHO|TTOS  |UND_TTO|Inv_Final_$|TTOS_$|USD_OH_TTOS|
+---------+------+--------+--------+------+---------+---------+---+------+-------+-----------+------+-----------+
|ENE      |1.0   |3656.0  |4711.0  |1539.0|1318.0   |

--------------


In [1]:
# ==============================================================================
# SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP - TODOS LOS PAÍSES Y FAMILIAS
# ==============================================================================
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, coalesce, greatest, round, lead, lag, row_number, desc,
    max as f_max, min as f_min, explode, array, count, countDistinct
)
from pyspark.sql.window import Window

print("="*80)
print("SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP - UNIVERSAL")
print("="*80)

# ==============================================================================
# PASO 1: CARGAR TODOS LOS DATOS DISPONIBLES (SIN FILTROS)
# ==============================================================================
print("\n[PASO 1] Cargando TODOS los datos del Lakehouse...")

df_master_basico_raw = spark.sql("""
    SELECT * 
    FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.continental_master_basic_daily
""")

print(f"✓ Registros cargados (raw): {df_master_basico_raw.count():,}")

# ==============================================================================
# PASO 2: ANÁLISIS DE DATOS DISPONIBLES
# ==============================================================================
print("\n[PASO 2] Analizando datos disponibles...")

print("\n📊 PAÍSES DISPONIBLES:")
df_master_basico_raw.select("PAIS").distinct().orderBy("PAIS").show(100, truncate=False)

print("\n📊 FAMILIAS DISPONIBLES:")
df_master_basico_raw.select("Familia").distinct().orderBy("Familia").show(100, truncate=False)

print("\n📊 AÑOS DISPONIBLES:")
df_master_basico_raw.select("Ano").distinct().orderBy("Ano").show(100, truncate=False)

print("\n📊 COMBINACIONES PAÍS-FAMILIA-AÑO:")
(df_master_basico_raw
 .groupBy("PAIS", "Familia", "Ano")
 .agg(F.count("*").alias("Registros"),
      F.min("ORDENM").alias("Mes_Min"),
      F.max("ORDENM").alias("Mes_Max"))
 .orderBy("PAIS", "Familia", "Ano")
 .show(100, truncate=False))

# ==============================================================================
# PASO 3: DEDUPLICACIÓN COMPLETA POR CLAVE PRIMARIA
# ==============================================================================
print("\n[PASO 3] Eliminando duplicados...")

# Definir ventana de deduplicación por clave primaria completa
w_dedup = Window.partitionBy("PAIS", "Familia", "Ano", "ORDENM", "PWOH").orderBy("Mes")

df_master_basico = (
    df_master_basico_raw
    .withColumn("rn", row_number().over(w_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

count_antes = df_master_basico_raw.count()
count_despues = df_master_basico.count()
duplicados = count_antes - count_despues

print(f"✓ Registros antes: {count_antes:,}")
print(f"✓ Registros después: {count_despues:,}")
print(f"✓ Duplicados eliminados: {duplicados:,}")

# ==============================================================================
# PASO 4: VALIDACIÓN DE CALIDAD DE DATOS
# ==============================================================================
print("\n[PASO 4] Validación de calidad de datos...")

print("\n🔍 Registros con datos nulos en columnas críticas:")
(df_master_basico
 .select(
     F.sum(when(col("Inv_Inicial").isNull(), 1).otherwise(0)).alias("Inv_Inicial_NULL"),
     F.sum(when(col("Llegadas").isNull(), 1).otherwise(0)).alias("Llegadas_NULL"),
     F.sum(when(col("Fcst").isNull(), 1).otherwise(0)).alias("Fcst_NULL"),
     F.sum(when(col("Facturacion").isNull(), 1).otherwise(0)).alias("Facturacion_NULL"),
     F.sum(when(col("Inv_Final").isNull(), 1).otherwise(0)).alias("Inv_Final_NULL")
 )
 .show(truncate=False))

# ==============================================================================
# PASO 5: MUESTRA DE DATOS POR PAÍS Y FAMILIA
# ==============================================================================
print("\n[PASO 5] Muestras de datos cargados...")

# Obtener lista de combinaciones únicas
combinaciones = (df_master_basico
                .select("PAIS", "Familia", "Ano")
                .distinct()
                .orderBy("PAIS", "Familia", "Ano")
                .collect())

print(f"\n✓ Se cargarán {len(combinaciones)} combinaciones PAIS-FAMILIA-AÑO")

# Mostrar muestra de las 3 primeras combinaciones
print("\n📋 MUESTRA DE LAS PRIMERAS 3 COMBINACIONES:")
for i, row in enumerate(combinaciones[:3]):
    pais = row['PAIS']
    familia = row['Familia']
    ano = row['Ano']
    
    print(f"\n--- MUESTRA {i+1}: {pais} - {familia} - {ano} ---")
    (df_master_basico
     .filter((col("PAIS") == pais) & 
             (col("Familia") == familia) & 
             (col("Ano") == ano))
     .select("Attribute", "ORDENM", 
             F.round("Inv_Inicial", 0).alias("Inv_Inic"),
             F.round("Llegadas", 0).alias("Llegadas"),
             F.round("Fcst", 0).alias("Fcst"),
             F.round("Facturacion", 0).alias("Fact"),
             F.round("Inv_Final", 0).alias("Inv_Final"))
     .orderBy("ORDENM")
     .show(12, truncate=False))

# ==============================================================================
# PASO 6: ESTADÍSTICAS FINALES
# ==============================================================================
print("\n[PASO 6] Estadísticas finales de carga...")

stats = (df_master_basico
         .agg(
             F.countDistinct("PAIS").alias("Total_Paises"),
             F.countDistinct("Familia").alias("Total_Familias"),
             F.countDistinct("Ano").alias("Total_Anos"),
             F.count("*").alias("Total_Registros"),
             F.sum(when(col("Fcst") > 0, 1).otherwise(0)).alias("Registros_con_Fcst"),
             F.sum(when(col("Facturacion") > 0, 1).otherwise(0)).alias("Registros_con_Fact")
         ))

print("\n📊 RESUMEN DE CARGA:")
stats.show(truncate=False)

# Cache para optimizar procesamiento posterior
df_master_basico = df_master_basico.cache()
print(f"\n✓ DataFrame cacheado en memoria")

print("\n" + "="*80)
print("✅ SNIPPET 1 COMPLETADO")
print("="*80)
print("RESUMEN:")
print(f"  • Países: {stats.first()['Total_Paises']}")
print(f"  • Familias: {stats.first()['Total_Familias']}")
print(f"  • Años: {stats.first()['Total_Anos']}")
print(f"  • Total registros: {stats.first()['Total_Registros']:,}")
print("="*80)

StatementMeta(, 5fc90911-6e7a-4e9b-8780-73c216b9965d, 3, Finished, Available, Finished)

SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP - UNIVERSAL

[PASO 1] Cargando TODOS los datos del Lakehouse...
✓ Registros cargados (raw): 4,608

[PASO 2] Analizando datos disponibles...

📊 PAÍSES DISPONIBLES:
+----+
|PAIS|
+----+
|AR  |
|CA  |
|CL  |
|CO  |
|CR  |
|EC  |
|GT  |
|HN  |
|PE  |
|SV  |
+----+


📊 FAMILIAS DISPONIBLES:
+-------+
|Familia|
+-------+
|ACCE   |
|ACCESOR|
|AIAC   |
|CALE   |
|COCC   |
|GLOB   |
|LAVA   |
|REFR   |
+-------+


📊 AÑOS DISPONIBLES:
+----+
|Ano |
+----+
|2025|
|2026|
|2027|
|2028|
|2029|
|2030|
+----+


📊 COMBINACIONES PAÍS-FAMILIA-AÑO:
+----+-------+----+---------+-------+-------+
|PAIS|Familia|Ano |Registros|Mes_Min|Mes_Max|
+----+-------+----+---------+-------+-------+
|AR  |COCC   |2025|12       |1      |12     |
|AR  |COCC   |2026|12       |1      |12     |
|AR  |COCC   |2027|12       |1      |12     |
|AR  |COCC   |2028|12       |1      |12     |
|AR  |COCC   |2029|12       |1      |12     |
|AR  |COCC   |2030|12       |1      |12     |
|AR  |GLOB   

In [2]:
# ==============================================================================
# SNIPPET 2: DETECCIÓN DEL MES ANCLA GLOBAL (UN SOLO ANCLA)
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 2: DETECCIÓN DEL MES ANCLA GLOBAL")
print("="*80)

# Ventana GLOBAL (sin particiones)
w_global = Window.orderBy(lit(1))

df_con_ancla = (
    df_master_basico
    # Periodo único para comparar
    .withColumn("periodo_id", col("Ano") * 100 + col("ORDENM"))
    
    # Detectar último mes con facturación > 0 EN TODA LA TABLA
    .withColumn("periodo_candidato", 
                when(coalesce(col("Facturacion"), lit(0.0)) > 0.0, col("periodo_id")))
    .withColumn("periodo_ancla", f_max("periodo_candidato").over(w_global))
    
    # Fallback
    .withColumn("periodo_ancla", 
                coalesce(col("periodo_ancla"), lit(202501)))
    
    # Extraer año y mes
    .withColumn("ano_ancla", (col("periodo_ancla") / 100).cast("int"))
    .withColumn("mes_ancla", col("periodo_ancla") % 100)
    
    # Flags
    .withColumn("es_mes_ancla", 
                (col("Ano") == col("ano_ancla")) & (col("ORDENM") == col("mes_ancla")))
    .withColumn("antes_del_ancla", col("periodo_id") < col("periodo_ancla"))
    .withColumn("despues_del_ancla", col("periodo_id") > col("periodo_ancla"))
    
    .drop("periodo_candidato")
)

# Verificación
print("\n🎯 ANCLA GLOBAL DETECTADA:")
df_con_ancla.select("ano_ancla", "mes_ancla", "periodo_ancla").distinct().show()

print("\n✓ Meses marcados como ancla:")
df_con_ancla.filter(col("es_mes_ancla")).select("PAIS", "Familia", "Ano", "ORDENM").show(10)

count_ancla = df_con_ancla.filter(col("es_mes_ancla")).count()
print(f"\n✓ Total registros en mes ancla: {count_ancla}")

print("\n[✓] SNIPPET 2 COMPLETADO")

StatementMeta(, 5fc90911-6e7a-4e9b-8780-73c216b9965d, 4, Finished, Available, Finished)


SNIPPET 2: DETECCIÓN DEL MES ANCLA GLOBAL

🎯 ANCLA GLOBAL DETECTADA:
+---------+---------+-------------+
|ano_ancla|mes_ancla|periodo_ancla|
+---------+---------+-------------+
|     2025|       12|       202512|
+---------+---------+-------------+


✓ Meses marcados como ancla:
+----+-------+----+------+
|PAIS|Familia| Ano|ORDENM|
+----+-------+----+------+
|  CA|   CALE|2025|    12|
|  EC|   CALE|2025|    12|
|  CL|   COCC|2025|    12|
|  CO|   CALE|2025|    12|
|  GT|   CALE|2025|    12|
|  CA|   ACCE|2025|    12|
|  PE|   GLOB|2025|    12|
|  AR|   REFR|2025|    12|
|  CR|   AIAC|2025|    12|
|  GT|   COCC|2025|    12|
+----+-------+----+------+
only showing top 10 rows


✓ Total registros en mes ancla: 64

[✓] SNIPPET 2 COMPLETADO


In [3]:
# ==============================================================================
# SNIPPET 3: CÁLCULO LLEGADAS E INVENTARIOS - PROPAGACIÓN MES A MES
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 3: MOTOR DE CÁLCULO - PROPAGACIÓN MES A MES")
print("="*80)

# Ventana por PAIS-Familia-PWOH ordenada por Ano-Mes
w_continuo = Window.partitionBy("PAIS", "Familia", "PWOH").orderBy("Ano", "ORDENM")

# Ventana por PAIS-Familia-Año para detectar último S&OP
w_año = Window.partitionBy("PAIS", "Familia", "Ano")

# Ventana por PAIS-Familia-Año-PWOH para obtener S&OP de enero del mismo año
w_año_pwoh = Window.partitionBy("PAIS", "Familia", "Ano", "PWOH")

# ==============================================================================
# PASO 1: DETECTAR ÚLTIMO MES CON S&OP POR AÑO
# ==============================================================================
print("\n[PASO 1] Detectando último mes con S&OP por año-país-familia...")

df_con_limites = (
    df_con_ancla
    .withColumn("tiene_sop", when(coalesce(col("Fcst"), lit(0.0)) > 0.0, col("ORDENM")))
    .withColumn("ultimo_mes_sop", f_max("tiene_sop").over(w_año))
    .withColumn("ultimo_mes_sop", coalesce(col("ultimo_mes_sop"), lit(12)))
    .drop("tiene_sop")
)

print("\n📊 Límites S&OP detectados:")
(df_con_limites
 .select("PAIS", "Familia", "Ano", "ultimo_mes_sop")
 .distinct()
 .orderBy("PAIS", "Familia", "Ano")
 .show(50, truncate=False))

# ==============================================================================
# PASO 2: EXPANDIR PWOH 1-10 Y CREAR FLAGS PERMANENTES
# ==============================================================================
print("\n[PASO 2] Expandiendo PWOH 1-10 y creando flags...")

df_con_pwoh = (
    df_con_limites
    # FLAGS PERMANENTES
    .withColumn("llegadas_original_daily", coalesce(col("Llegadas"), lit(0.0)))
    .withColumn("tiene_llegadas_daily", coalesce(col("Llegadas"), lit(0.0)) > 0.0)
    .withColumn("PWOH", explode(array([lit(i) for i in range(1, 11)])))
    
    # Capturar S&OP de ENERO del mismo año para usar en DICIEMBRE
    .withColumn("sop_enero_mismo_año", 
                f_max(when(col("ORDENM") == 1, col("Fcst"))).over(w_año_pwoh))
)

print(f"✓ Registros después de expandir PWOH: {df_con_pwoh.count():,}")

# ==============================================================================
# PASO 3: CAPTURAR INV_FINAL DEL MES ANCLA
# ==============================================================================
print("\n[PASO 3] Capturando Inv_Final del mes ancla...")

w_ancla = Window.partitionBy("PAIS", "Familia")

df_preparado = (
    df_con_pwoh
    .withColumn("inv_final_ancla",
                f_max(when(col("es_mes_ancla"), col("Inv_Final"))).over(w_ancla))
    .withColumn("es_primer_mes_despues",
                col("periodo_id") == (col("periodo_ancla") + 1))
)

# ==============================================================================
# PASO 4: ITERACIÓN PARA PROPAGACIÓN MES A MES
# ==============================================================================
print("\n[PASO 4] Iniciando propagación mes a mes (15 iteraciones)...")

df_calculado = df_preparado

for iteracion in range(1, 16):
    print(f"  Iteración {iteracion}/15...")
    
    # A. INV_INICIAL: tomar del mes anterior
    df_calculado = (
        df_calculado
        .withColumn("Inv_Final_anterior", lag("Inv_Final").over(w_continuo))
        .withColumn(
            "Inv_Inicial_nuevo",
            when(
                # ANTES DEL ANCLA O MES ANCLA: mantener del Daily
                col("antes_del_ancla") | col("es_mes_ancla"),
                col("Inv_Inicial")
            ).when(
                # PRIMER MES DESPUÉS ANCLA: heredar Inv_Final del ancla
                col("es_primer_mes_despues"),
                coalesce(col("inv_final_ancla"), lit(0.0))
            ).otherwise(
                # RESTO: tomar Inv_Final del mes anterior (lag)
                coalesce(col("Inv_Final_anterior"), col("Inv_Inicial"))
            )
        )
        .drop("Inv_Final_anterior")
    )
    
    # B. LLEGADAS: CALCULAR SEGÚN POSICIÓN Y CONDICIONES
    df_calculado = (
        df_calculado
        .withColumn("S&OP_siguiente", lead("Fcst", 1).over(w_continuo))
        .withColumn(
            "Llegadas_nuevo",
            when(
                # ANTES DEL ANCLA: CALCULAR con despeje (Inv_Final - Inv_Inicial + S&OP)
                col("antes_del_ancla") & (coalesce(col("Fcst"), lit(0.0)) > 0.0),
                greatest(
                    lit(0.0),
                    coalesce(col("Inv_Final"), lit(0.0)) - 
                    coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) + 
                    coalesce(col("Fcst"), lit(0.0))
                )
            ).when(
                # MES ANCLA: mantener del Daily
                col("es_mes_ancla"),
                col("llegadas_original_daily")
            ).when(
                # DESPUÉS ANCLA - HAY llegadas en Daily: mantenerlas
                col("despues_del_ancla") & col("tiene_llegadas_daily"),
                col("llegadas_original_daily")
            ).when(
                # DESPUÉS ANCLA - NO hay llegadas - MES 12: usar S&OP de ENERO del mismo año
                col("despues_del_ancla") & 
                ~col("tiene_llegadas_daily") &
                (col("ORDENM") == 12) &
                (coalesce(col("Fcst"), lit(0.0)) > 0.0),
                greatest(
                    lit(0.0),
                    ((coalesce(col("sop_enero_mismo_año"), lit(0.0)) / 4.0) * col("PWOH"))
                    + coalesce(col("Fcst"), lit(0.0))
                    - coalesce(col("Inv_Inicial_nuevo"), lit(0.0))
                )
            ).when(
                # DESPUÉS ANCLA - NO hay llegadas - RESTO DE MESES: usar S&OP siguiente
                col("despues_del_ancla") & 
                ~col("tiene_llegadas_daily") &
                (col("ORDENM") <= col("ultimo_mes_sop")) &
                (coalesce(col("Fcst"), lit(0.0)) > 0.0),
                greatest(
                    lit(0.0),
                    ((coalesce(col("S&OP_siguiente"), lit(0.0)) / 4.0) * col("PWOH"))
                    + coalesce(col("Fcst"), lit(0.0))
                    - coalesce(col("Inv_Inicial_nuevo"), lit(0.0))
                )
            ).otherwise(lit(0.0))
        )
        .drop("S&OP_siguiente")
    )
    
    # C. INV_FINAL: Inv_Inicial + Llegadas - S&OP
    df_calculado = df_calculado.withColumn(
        "Inv_Final_nuevo",
        when(
            # ANTES DEL ANCLA O MES ANCLA: mantener del Daily
            col("antes_del_ancla") | col("es_mes_ancla"),
            coalesce(col("Inv_Final"), lit(0.0))
        ).when(
            # DESPUÉS DEL ANCLA: calcular
            col("despues_del_ancla") &
            (col("ORDENM") <= col("ultimo_mes_sop")) &
            (coalesce(col("Fcst"), lit(0.0)) > 0.0),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0))
                + coalesce(col("Llegadas_nuevo"), lit(0.0))
                - coalesce(col("Fcst"), lit(0.0))
            )
        ).otherwise(lit(0.0))
    )
    
    # D. ACTUALIZAR COLUMNAS
    df_calculado = (
        df_calculado
        .drop("Inv_Inicial", "Llegadas", "Inv_Final")
        .withColumnRenamed("Inv_Inicial_nuevo", "Inv_Inicial")
        .withColumnRenamed("Llegadas_nuevo", "Llegadas")
        .withColumnRenamed("Inv_Final_nuevo", "Inv_Final")
    )

# ==============================================================================
# PASO 5: FILTRAR Y LIMPIAR
# ==============================================================================
print("\n[PASO 5] Filtrando registros hasta último mes con S&OP...")

df_calculado = (
    df_calculado
    .filter(col("ORDENM") <= col("ultimo_mes_sop"))
    .drop("sop_enero_mismo_año")  # Limpiar columna temporal
)

print(f"✓ Registros después de filtrar: {df_calculado.count():,}")

# Cache final
df_calculado = df_calculado.cache()

print("\n[✓] SNIPPET 3 COMPLETADO")

# ==============================================================================
# DEBUG
# ==============================================================================
print("\n" + "="*80)
print("DEBUG - CR LAVA PWOH=3")
print("="*80)

print("\n📊 PWOH=3 - TODOS LOS MESES:")
(df_calculado
 .filter((col("PAIS") == "CR") & 
         (col("Familia") == "LAVA") & 
         (col("PWOH") == 3))
 .select(
     col("Ano"),
     col("ORDENM"),
     F.round("Inv_Inicial", 0).alias("Inv_Inic"),
     F.round("Llegadas", 0).alias("Llegadas"),
     F.round("Fcst", 0).alias("S&OP"),
     F.round("Inv_Final", 0).alias("Inv_Final"),
     col("tiene_llegadas_daily").alias("Del_Daily")
 )
 .orderBy("Ano", "ORDENM")
 .show(30, truncate=False))

print("\n[✓] DEBUG COMPLETADO")

StatementMeta(, 5fc90911-6e7a-4e9b-8780-73c216b9965d, 5, Finished, Available, Finished)


SNIPPET 3: MOTOR DE CÁLCULO - PROPAGACIÓN MES A MES

[PASO 1] Detectando último mes con S&OP por año-país-familia...

📊 Límites S&OP detectados:
+----+-------+----+--------------+
|PAIS|Familia|Ano |ultimo_mes_sop|
+----+-------+----+--------------+
|AR  |COCC   |2025|12            |
|AR  |COCC   |2026|12            |
|AR  |COCC   |2027|12            |
|AR  |COCC   |2028|12            |
|AR  |COCC   |2029|12            |
|AR  |COCC   |2030|12            |
|AR  |GLOB   |2025|12            |
|AR  |GLOB   |2026|12            |
|AR  |GLOB   |2027|12            |
|AR  |GLOB   |2028|12            |
|AR  |GLOB   |2029|12            |
|AR  |GLOB   |2030|12            |
|AR  |LAVA   |2025|12            |
|AR  |LAVA   |2026|12            |
|AR  |LAVA   |2027|12            |
|AR  |LAVA   |2028|12            |
|AR  |LAVA   |2029|12            |
|AR  |LAVA   |2030|12            |
|AR  |REFR   |2025|12            |
|AR  |REFR   |2026|12            |
|AR  |REFR   |2027|12            |
|AR  |REFR   |

In [4]:
# ==============================================================================
# SNIPPET 4: CÁLCULO DE TTOS
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 4: CÁLCULO DE TTOS")
print("="*80)

print("\n📋 FÓRMULA: TTOS = Llegadas[t+1] + (Llegadas[t+2] / 2)")

# Ventanas para obtener llegadas futuras
w_intra_ano = Window.partitionBy("PAIS", "Familia", "Ano", "PWOH").orderBy("ORDENM")
w_multi_ano = Window.partitionBy("PAIS", "Familia", "PWOH").orderBy("Ano", "ORDENM")

df_ttos = (
    df_calculado  # ← CORRECCIÓN: era df_con_woh
    # Llegadas mes siguiente (dentro del año)
    .withColumn("Lleg_t1_intra", lead("Llegadas", 1).over(w_intra_ano))
    # Llegadas 2 meses adelante (dentro del año)
    .withColumn("Lleg_t2_intra", lead("Llegadas", 2).over(w_intra_ano))
    # Llegadas mes siguiente (cruzando años)
    .withColumn("Lleg_t1_multi", lead("Llegadas", 1).over(w_multi_ano))
    # Llegadas 2 meses adelante (cruzando años)
    .withColumn("Lleg_t2_multi", lead("Llegadas", 2).over(w_multi_ano))
    
    # TTOS según casos
    .withColumn(
        "TTOS",
        when(
            # NOVIEMBRE: usa siguiente intra + 2do puede cruzar año
            col("ORDENM") == 11,
            coalesce(col("Lleg_t1_intra"), lit(0.0)) +
            (coalesce(col("Lleg_t2_multi"), col("Lleg_t1_intra"), lit(0.0)) / 2.0)
        ).when(
            # DICIEMBRE: ambos cruzan año
            col("ORDENM") == 12,
            coalesce(col("Lleg_t1_multi"), lit(0.0)) +
            (coalesce(col("Lleg_t2_multi"), lit(0.0)) / 2.0)
        ).otherwise(
            # RESTO: todo intra-año
            coalesce(col("Lleg_t1_intra"), lit(0.0)) +
            (coalesce(col("Lleg_t2_intra"), lit(0.0)) / 2.0)
        )
    )
    
    # UND_TTOS = Inv_Final + TTOS
    .withColumn("UND_TTOS", F.round(col("Inv_Final") + col("TTOS"), 0))
    
    # Limpiar columnas temporales
    .drop("Lleg_t1_intra", "Lleg_t2_intra", "Lleg_t1_multi", "Lleg_t2_multi")
)

df_ttos = df_ttos.cache()

print(f"\n✓ TTOS calculado: {df_ttos.count():,} registros")

# Debug
print("\n📊 SAMPLE CR-LAVA PWOH=1 - 2025 con TTOS:")
(df_ttos
 .filter((col("PAIS") == "CR") & 
         (col("Familia") == "LAVA") & 
         (col("PWOH") == 1) &
         (col("Ano") == 2025))
 .select(
     col("Attribute"),
     col("ORDENM"),
     F.round("Inv_Final", 0).alias("Inv_Final"),
     F.round("Llegadas", 0).alias("Llegadas"),
     F.round("TTOS", 0).alias("TTOS"),
     F.round("UND_TTOS", 0).alias("UND_TTOS")
 )
 .orderBy("ORDENM")
 .show(12, truncate=False))

print("\n[✓] SNIPPET 4 COMPLETADO")

StatementMeta(, 5fc90911-6e7a-4e9b-8780-73c216b9965d, 6, Finished, Available, Finished)


SNIPPET 4: CÁLCULO DE TTOS

📋 FÓRMULA: TTOS = Llegadas[t+1] + (Llegadas[t+2] / 2)

✓ TTOS calculado: 45,970 registros

📊 SAMPLE CR-LAVA PWOH=1 - 2025 con TTOS:
+---------+------+---------+--------+------+--------+
|Attribute|ORDENM|Inv_Final|Llegadas|TTOS  |UND_TTOS|
+---------+------+---------+--------+------+--------+
|ENE      |1     |7049.0   |4932.0  |3751.0|10800.0 |
|FEB      |2     |9011.0   |3501.0  |499.0 |9510.0  |
|MAR      |3     |8191.0   |499.0   |27.0  |8218.0  |
|ABR      |4     |5782.0   |0.0     |1424.0|7206.0  |
|MAY      |5     |4139.0   |53.0    |2741.0|6880.0  |
|JUN      |6     |4174.0   |2741.0  |1554.0|5728.0  |
|JUL      |7     |2501.0   |0.0     |3648.0|6149.0  |
|AGO      |8     |3495.0   |3108.0  |1979.0|5474.0  |
|SEP      |9     |2916.0   |1080.0  |2143.0|5059.0  |
|OCT      |10    |2943.0   |1797.0  |2296.0|5239.0  |
|NOV      |11    |1845.0   |691.0   |3826.0|5671.0  |
|DIC      |12    |2074.0   |3209.0  |1994.0|4068.0  |
+---------+------+---------+-

In [5]:
# ==============================================================================
# SNIPPET 5: CÁLCULO DE WOH Y WOH_TTOS
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 5: CÁLCULO DE WOH Y WOH_TTOS")
print("="*80)

print("\n📋 LÓGICA:")
print("- WOH = Inv_Final / (Promedio_Semanal_Forecast)")
print("- Antes último facturado: 8 semanas")
print("- Después último facturado: 4 semanas")

# Detectar último mes con facturación
df_ultimo_fact = (
    df_ttos
    .filter(col("Facturacion") > 0)
    .groupBy("PAIS", "Familia", "Ano", "PWOH")
    .agg(f_max("ORDENM").alias("ultimo_fact"))
)

# Fallbacks enero/febrero 2025
df_fallbacks = (
    df_ttos
    .filter((col("Ano") == 2025) & col("ORDENM").isin(1, 2))
    .groupBy("PAIS", "Familia", "PWOH")
    .pivot("ORDENM", [1, 2])
    .agg(F.first("Fcst"))
    .withColumnRenamed("1", "fcst_ene_2025")
    .withColumnRenamed("2", "fcst_feb_2025")
)

# Ventana intra-año
w_intra_woh = Window.partitionBy("PAIS", "Familia", "Ano", "PWOH").orderBy("ORDENM")

df_woh = (
    df_ttos
    .join(df_ultimo_fact, ["PAIS", "Familia", "Ano", "PWOH"], "left")
    .join(df_fallbacks, ["PAIS", "Familia", "PWOH"], "left")
    .withColumn("ultimo_fact", coalesce(col("ultimo_fact"), lit(0)))
    .fillna(0, ["fcst_ene_2025", "fcst_feb_2025"])
    .withColumn("fcst_m1", lead("Fcst", 1).over(w_intra_woh))
    .withColumn("fcst_m2", lead("Fcst", 2).over(w_intra_woh))
    
    # Denominador inteligente
    .withColumn(
        "denominador",
        when(
            col("ORDENM") <= col("ultimo_fact"),  # 8 semanas
            when(
                col("ORDENM") == 11,  # NOV
                (coalesce(col("fcst_m1"), lit(0.0)) + 
                 coalesce(col("fcst_ene_2025"), lit(0.0))) / 8.0
            ).when(
                col("ORDENM") == 12,  # DIC
                (coalesce(col("fcst_ene_2025"), lit(0.0)) + 
                 coalesce(col("fcst_feb_2025"), lit(0.0))) / 8.0
            ).otherwise(
                (coalesce(col("fcst_m1"), lit(0.0)) + 
                 coalesce(col("fcst_m2"), lit(0.0))) / 8.0
            )
        ).otherwise(  # 4 semanas
            when(
                col("ORDENM") == 12,  # DIC
                coalesce(col("fcst_ene_2025"), lit(0.0)) / 4.0
            ).otherwise(
                coalesce(col("fcst_m1"), lit(0.0)) / 4.0
            )
        )
    )
    
    # WOH como enteros
    .withColumn(
        "WOH",
        when((col("Inv_Final") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("Inv_Final") / col("denominador"), 0).cast("int"))
    )
    .withColumn(
        "WOH_TTOS",
        when((col("UND_TTOS") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("UND_TTOS") / col("denominador"), 0).cast("int"))
    )
    
    # Renombrar Inv_Inic para consistencia con snippet 6
    .withColumnRenamed("Inv_Inicial", "Inv_Inic")
    
    .drop("fcst_m1", "fcst_m2", "denominador", "fcst_ene_2025", "fcst_feb_2025")
)

df_woh = df_woh.cache()

print(f"\n✓ WOH calculado: {df_woh.count():,} registros")

# Debug
print("\n📊 SAMPLE CR-LAVA PWOH=1 - 2025 con WOH:")
(df_woh
 .filter((col("PAIS") == "CR") & 
         (col("Familia") == "LAVA") & 
         (col("PWOH") == 3) &
         (col("Ano") == 2026))
 .select(
     col("Attribute"),
     col("ORDENM"),
     F.round("Inv_Final", 0).alias("Inv_Final"),
     F.round("UND_TTOS", 0).alias("UND_TTOS"),
     col("WOH"),
     col("WOH_TTOS")
 )
 .orderBy("ORDENM")
 .show(12, truncate=False))

print("\n[✓] SNIPPET 5 COMPLETADO")

StatementMeta(, 5fc90911-6e7a-4e9b-8780-73c216b9965d, 7, Finished, Available, Finished)


SNIPPET 5: CÁLCULO DE WOH Y WOH_TTOS

📋 LÓGICA:
- WOH = Inv_Final / (Promedio_Semanal_Forecast)
- Antes último facturado: 8 semanas
- Después último facturado: 4 semanas

✓ WOH calculado: 45,970 registros

📊 SAMPLE CR-LAVA PWOH=1 - 2025 con WOH:
+---------+------+---------+--------+---+--------+
|Attribute|ORDENM|Inv_Final|UND_TTOS|WOH|WOH_TTOS|
+---------+------+---------+--------+---+--------+
|ENE      |1     |1105.0   |3271.0  |2  |7       |
|FEB      |2     |830.0    |4322.0  |1  |6       |
|MAR      |3     |0.0      |5358.0  |0  |9       |
|ABR      |4     |2010.0   |5156.0  |3  |8       |
|MAY      |5     |1233.0   |4933.0  |3  |12      |
|JUN      |6     |2075.0   |5414.0  |3  |8       |
|JUL      |7     |1736.0   |4675.0  |3  |8       |
|AGO      |8     |1244.0   |5401.0  |3  |13      |
|SEP      |9     |1819.0   |7107.0  |3  |12      |
|OCT      |10    |3239.0   |7251.0  |3  |7       |
|NOV      |11    |1805.0   |4059.0  |3  |7       |
|DIC      |12    |1652.0   |1652.0  |4 

In [6]:
# ==============================================================================
# SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL (CON AGREGACIÓN REGIONAL)
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL")
print("="*80)

# ==============================================================================
# PASO 1: CARGAR ARCHIVOS BUDGET
# ==============================================================================
print("\n[PASO 1] Cargando archivos de budget...")

try:
    df_bgt_pp = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_PP.csv")
        .select(col("Pais").alias("PAIS"), col("Familia"), 
                col("PP").cast("double").alias("BGT_PP"))
    )
    print(f"✓ BGT_PP cargado exitosamente: {df_bgt_pp.count():,} registros")
except:
    print("⚠ BGT_PP no encontrado, usando defaults")
    df_bgt_pp = spark.createDataFrame([
        ("CR", "LAVA", 164.5), ("CO", "LAVA", 100.0), ("MX", "SAGA", 120.0)
    ], ["PAIS", "Familia", "BGT_PP"])

try:
    df_bgt_cont = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_CONT.csv")
        .select("PAIS", "Familia", "Attribute", col("ORDENM").cast("int"), 
                col("BGT_QTY").cast("double"), 
                col("BGT_$").cast("double").alias("BGT_USD"))
    )
    print(f"✓ BGT_CONT cargado exitosamente: {df_bgt_cont.count():,} registros")
except:
    print("⚠ BGT_CONT no encontrado, usando defaults")
    df_bgt_cont = spark.createDataFrame([
        ("CR", "LAVA", "ENE", 1, 2550.0, 100000.0),
        ("CR", "LAVA", "FEB", 2, 2219.0, 95000.0),
        ("CO", "LAVA", "ENE", 1, 1000.0, 80000.0),
        ("MX", "SAGA", "ENE", 1, 1500.0, 120000.0)
    ], ["PAIS", "Familia", "Attribute", "ORDENM", "BGT_QTY", "BGT_USD"])

# ==============================================================================
# PASO 2: CALCULAR MONETARIOS
# ==============================================================================
print("\n[PASO 2] Calculando campos monetarios...")

df_con_monetarios = (
    df_woh
    .join(df_bgt_pp, ["PAIS", "Familia"], "left")
    .join(df_bgt_cont, ["PAIS", "Familia", "Attribute", "ORDENM"], "left")
    .fillna(0, ["BGT_PP", "BGT_QTY", "BGT_USD"])
    
    # Columnas monetarias (en millones)
    .withColumn("Inv_Final_USD", (col("Inv_Final") * col("BGT_PP")) / 1_000_000)
    .withColumn("TTOS_USD", (col("TTOS") * col("BGT_PP")) / 1_000_000)
    .withColumn("USD_OH_TTOS", col("Inv_Final_USD") + col("TTOS_USD"))
    
    # Diferencias vs budget
    .withColumn("Dif_Real_vs_BGT_QTY", col("UND_TTOS") - col("BGT_QTY"))
    .withColumn("Dif_Real_vs_BGT_USD", col("USD_OH_TTOS") - col("BGT_USD"))
)

print(f"✓ Monetarios agregados: {df_con_monetarios.count():,} registros")

# ==============================================================================
# PASO 3: CREAR TABLA FINAL PARA POWER BI
# ==============================================================================
print("\n[PASO 3] Creando tabla final con estructura Power BI...")

df_tabla_final = (
    df_con_monetarios
    .select(
        col("Ano").cast("int"),
        col("Mes"),
        when(col("PAIS").isin("CO", "EC", "PE", "AR", "CL"), "Andino")
        .when(col("PAIS").isin("CR", "GT", "HN", "SV"), "CEAM")
        .when(col("PAIS").isin("MX"), "Mexico")
        .when(col("PAIS").isin("CA"), "NorteAmerica")
        .otherwise("Otros").alias("Region"),
        col("PAIS"),
        col("Familia"),
        col("Attribute"),
        col("Inv_Inic").cast("double"),
        col("Llegadas").cast("double"),
        col("Fcst").alias("S&PO").cast("double"),
        col("Facturacion").alias("Facturado").cast("double"),
        col("Inv_Final").cast("double"),
        F.round(col("Inv_Final_USD"), 4).alias("Inv_Final_$"),
        col("WOH").alias("WHO").cast("int"),
        col("TTOS").cast("double"),
        F.round(col("TTOS_USD"), 4).alias("TTOS_$"),
        F.round(col("USD_OH_TTOS"), 4).alias("USD_OH_TTOS"),
        col("UND_TTOS").alias("UND_TTO").cast("double"),
        col("WOH_TTOS").alias("WOH_TTO").cast("int"),
        col("BGT_QTY").cast("double"),
        col("BGT_USD").alias("BGT_$").cast("double"),
        col("Dif_Real_vs_BGT_QTY").cast("double"),
        col("Dif_Real_vs_BGT_USD").alias("Dif_Real_vs_BGT_$").cast("double"),
        col("PWOH").cast("double"),
        col("ORDENM").cast("double"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Tabla final estructurada: {df_tabla_final.count():,} registros")

# ==============================================================================
# PASO 4: ELIMINAR DUPLICADOS
# ==============================================================================
print("\n[PASO 4] Eliminando duplicados por clave primaria...")

count_antes = df_tabla_final.count()

df_tabla_sin_duplicados = df_tabla_final.dropDuplicates([
    "PAIS", "Familia", "Ano", "ORDENM", "PWOH"
])

count_despues = df_tabla_sin_duplicados.count()
duplicados_eliminados = count_antes - count_despues

print(f"✓ Registros antes: {count_antes:,}")
print(f"✓ Registros después: {count_despues:,}")
print(f"✓ Duplicados eliminados: {duplicados_eliminados:,}")

# ==============================================================================
# PASO 4.5: AGREGACIÓN REGIONAL POR FAMILIA
# ==============================================================================
print("\n[PASO 4.5] Creando agregación regional por familia...")

df_regional = (
    df_tabla_sin_duplicados
    .groupBy("Region", "Familia", "Ano", "Mes", "Attribute", "ORDENM", "PWOH")
    .agg(
        F.sum("Inv_Inic").alias("Inv_Inic"),
        F.sum("Llegadas").alias("Llegadas"),
        F.sum("S&PO").alias("S&PO"),
        F.sum("Facturado").alias("Facturado"),
        F.sum("Inv_Final").alias("Inv_Final"),
        F.sum("Inv_Final_$").alias("Inv_Final_$"),
        F.sum("TTOS").alias("TTOS"),
        F.sum("TTOS_$").alias("TTOS_$"),
        F.sum("USD_OH_TTOS").alias("USD_OH_TTOS"),
        F.sum("UND_TTO").alias("UND_TTO"),
        F.sum("BGT_QTY").alias("BGT_QTY"),
        F.sum("BGT_$").alias("BGT_$"),
        F.sum("Dif_Real_vs_BGT_QTY").alias("Dif_Real_vs_BGT_QTY"),
        F.sum("Dif_Real_vs_BGT_$").alias("Dif_Real_vs_BGT_$")
    )
)

w_reg = Window.partitionBy("Region", "Familia", "PWOH").orderBy("Ano", "ORDENM")

df_regional = (
    df_regional
    .withColumn("fcst_m1", lead("S&PO", 1).over(w_reg))
    .withColumn("fcst_m2", lead("S&PO", 2).over(w_reg))
)

df_ult_fact_reg = (
    df_regional
    .filter(col("Facturado") > 0)
    .groupBy("Region", "Familia", "Ano", "PWOH")
    .agg(F.max("ORDENM").alias("ult_fact"))
)

df_regional = (
    df_regional
    .join(df_ult_fact_reg, ["Region", "Familia", "Ano", "PWOH"], "left")
    .withColumn("ult_fact", coalesce(col("ult_fact"), lit(0)))
    .withColumn(
        "denom",
        when(col("ORDENM") <= col("ult_fact"),
             (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0)
        .otherwise(coalesce(col("fcst_m1"), lit(0.0)) / 4.0)
    )
    .withColumn("WHO", 
                when(col("denom") > 0, F.round(col("Inv_Final") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .withColumn("WOH_TTO", 
                when(col("denom") > 0, F.round(col("UND_TTO") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .drop("fcst_m1", "fcst_m2", "denom", "ult_fact")
)

df_regional_final = (
    df_regional
    .select(
        col("Ano"), col("Mes"), col("Region"),
        col("Region").alias("PAIS"),
        col("Familia"),
        col("Attribute"), col("Inv_Inic"), col("Llegadas"), col("S&PO"), col("Facturado"),
        col("Inv_Final"), col("Inv_Final_$"), col("WHO"), col("TTOS"), col("TTOS_$"),
        col("USD_OH_TTOS"), col("UND_TTO"), col("WOH_TTO"), col("BGT_QTY"), col("BGT_$"),
        col("Dif_Real_vs_BGT_QTY"), col("Dif_Real_vs_BGT_$"), col("PWOH"), col("ORDENM"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Agregación regional por familia: {df_regional_final.count():,} registros")

# ==============================================================================
# PASO 4.6: AGREGACIÓN REGIONAL TODAS LAS FAMILIAS
# ==============================================================================
print("\n[PASO 4.6] Creando agregación regional TODAS las familias...")

df_regional_todas = (
    df_tabla_sin_duplicados
    .groupBy("Region", "Ano", "Mes", "Attribute", "ORDENM", "PWOH")  # SIN Familia
    .agg(
        F.sum("Inv_Inic").alias("Inv_Inic"),
        F.sum("Llegadas").alias("Llegadas"),
        F.sum("S&PO").alias("S&PO"),
        F.sum("Facturado").alias("Facturado"),
        F.sum("Inv_Final").alias("Inv_Final"),
        F.sum("Inv_Final_$").alias("Inv_Final_$"),
        F.sum("TTOS").alias("TTOS"),
        F.sum("TTOS_$").alias("TTOS_$"),
        F.sum("USD_OH_TTOS").alias("USD_OH_TTOS"),
        F.sum("UND_TTO").alias("UND_TTO"),
        F.sum("BGT_QTY").alias("BGT_QTY"),
        F.sum("BGT_$").alias("BGT_$"),
        F.sum("Dif_Real_vs_BGT_QTY").alias("Dif_Real_vs_BGT_QTY"),
        F.sum("Dif_Real_vs_BGT_$").alias("Dif_Real_vs_BGT_$")
    )
)

w_reg_todas = Window.partitionBy("Region", "PWOH").orderBy("Ano", "ORDENM")

df_regional_todas = (
    df_regional_todas
    .withColumn("fcst_m1", lead("S&PO", 1).over(w_reg_todas))
    .withColumn("fcst_m2", lead("S&PO", 2).over(w_reg_todas))
)

df_ult_fact_reg_todas = (
    df_regional_todas
    .filter(col("Facturado") > 0)
    .groupBy("Region", "Ano", "PWOH")
    .agg(F.max("ORDENM").alias("ult_fact"))
)

df_regional_todas = (
    df_regional_todas
    .join(df_ult_fact_reg_todas, ["Region", "Ano", "PWOH"], "left")
    .withColumn("ult_fact", coalesce(col("ult_fact"), lit(0)))
    .withColumn(
        "denom",
        when(col("ORDENM") <= col("ult_fact"),
             (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0)
        .otherwise(coalesce(col("fcst_m1"), lit(0.0)) / 4.0)
    )
    .withColumn("WHO", 
                when(col("denom") > 0, F.round(col("Inv_Final") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .withColumn("WOH_TTO", 
                when(col("denom") > 0, F.round(col("UND_TTO") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .drop("fcst_m1", "fcst_m2", "denom", "ult_fact")
)

df_regional_todas_final = (
    df_regional_todas
    .select(
        col("Ano"), col("Mes"), col("Region"),
        col("Region").alias("PAIS"),
        lit("Todas").alias("Familia"),  # ← FAMILIA = "Todas"
        col("Attribute"), col("Inv_Inic"), col("Llegadas"), col("S&PO"), col("Facturado"),
        col("Inv_Final"), col("Inv_Final_$"), col("WHO"), col("TTOS"), col("TTOS_$"),
        col("USD_OH_TTOS"), col("UND_TTO"), col("WOH_TTO"), col("BGT_QTY"), col("BGT_$"),
        col("Dif_Real_vs_BGT_QTY"), col("Dif_Real_vs_BGT_$"), col("PWOH"), col("ORDENM"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Agregación regional TODAS familias: {df_regional_todas_final.count():,} registros")

# ==============================================================================
# PASO 4.7: AGREGACIÓN POR PAÍS TODAS LAS FAMILIAS (NUEVO!)
# ==============================================================================
print("\n[PASO 4.7] Creando agregación por PAÍS TODAS las familias...")

df_pais_todas = (
    df_tabla_sin_duplicados
    .groupBy("PAIS", "Region", "Ano", "Mes", "Attribute", "ORDENM", "PWOH")  # SIN Familia
    .agg(
        F.sum("Inv_Inic").alias("Inv_Inic"),
        F.sum("Llegadas").alias("Llegadas"),
        F.sum("S&PO").alias("S&PO"),
        F.sum("Facturado").alias("Facturado"),
        F.sum("Inv_Final").alias("Inv_Final"),
        F.sum("Inv_Final_$").alias("Inv_Final_$"),
        F.sum("TTOS").alias("TTOS"),
        F.sum("TTOS_$").alias("TTOS_$"),
        F.sum("USD_OH_TTOS").alias("USD_OH_TTOS"),
        F.sum("UND_TTO").alias("UND_TTO"),
        F.sum("BGT_QTY").alias("BGT_QTY"),
        F.sum("BGT_$").alias("BGT_$"),
        F.sum("Dif_Real_vs_BGT_QTY").alias("Dif_Real_vs_BGT_QTY"),
        F.sum("Dif_Real_vs_BGT_$").alias("Dif_Real_vs_BGT_$")
    )
)

# Ventana por PAIS (en lugar de Region)
w_pais_todas = Window.partitionBy("PAIS", "PWOH").orderBy("Ano", "ORDENM")

df_pais_todas = (
    df_pais_todas
    .withColumn("fcst_m1", lead("S&PO", 1).over(w_pais_todas))
    .withColumn("fcst_m2", lead("S&PO", 2).over(w_pais_todas))
)

# Último mes con facturación por PAIS
df_ult_fact_pais_todas = (
    df_pais_todas
    .filter(col("Facturado") > 0)
    .groupBy("PAIS", "Ano", "PWOH")
    .agg(F.max("ORDENM").alias("ult_fact"))
)

df_pais_todas = (
    df_pais_todas
    .join(df_ult_fact_pais_todas, ["PAIS", "Ano", "PWOH"], "left")
    .withColumn("ult_fact", coalesce(col("ult_fact"), lit(0)))
    .withColumn(
        "denom",
        when(col("ORDENM") <= col("ult_fact"),
             (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0)
        .otherwise(coalesce(col("fcst_m1"), lit(0.0)) / 4.0)
    )
    .withColumn("WHO", 
                when(col("denom") > 0, F.round(col("Inv_Final") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .withColumn("WOH_TTO", 
                when(col("denom") > 0, F.round(col("UND_TTO") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .drop("fcst_m1", "fcst_m2", "denom", "ult_fact")
)

df_pais_todas_final = (
    df_pais_todas
    .select(
        col("Ano"), col("Mes"), col("Region"),
        col("PAIS"),  # ← Mantiene el PAIS real (México, Colombia, etc.)
        lit("Todas").alias("Familia"),  # ← FAMILIA = "Todas"
        col("Attribute"), col("Inv_Inic"), col("Llegadas"), col("S&PO"), col("Facturado"),
        col("Inv_Final"), col("Inv_Final_$"), col("WHO"), col("TTOS"), col("TTOS_$"),
        col("USD_OH_TTOS"), col("UND_TTO"), col("WOH_TTO"), col("BGT_QTY"), col("BGT_$"),
        col("Dif_Real_vs_BGT_QTY"), col("Dif_Real_vs_BGT_$"), col("PWOH"), col("ORDENM"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Agregación por PAÍS TODAS familias: {df_pais_todas_final.count():,} registros")

# ==============================================================================
# PASO 4.8: VALIDACIÓN DE DUPLICADOS EN AGREGACIONES (NUEVO!)
# ==============================================================================
print("\n[PASO 4.8] Validando que no existan duplicados en agregaciones...")

print("\n🔍 Verificando duplicados en df_regional_final:")
df_regional_dup_check = (
    df_regional_final
    .groupBy("PAIS", "Familia", "Ano", "ORDENM", "PWOH")
    .agg(F.count("*").alias("count"))
    .filter(col("count") > 1)
)
count_dup_regional = df_regional_dup_check.count()
if count_dup_regional > 0:
    print(f"⚠️  ADVERTENCIA: {count_dup_regional} duplicados encontrados en regional")
    df_regional_dup_check.show(10)
else:
    print("✓ No hay duplicados en agregación regional")

print("\n🔍 Verificando duplicados en df_regional_todas_final:")
df_regional_todas_dup_check = (
    df_regional_todas_final
    .groupBy("PAIS", "Familia", "Ano", "ORDENM", "PWOH")
    .agg(F.count("*").alias("count"))
    .filter(col("count") > 1)
)
count_dup_regional_todas = df_regional_todas_dup_check.count()
if count_dup_regional_todas > 0:
    print(f"⚠️  ADVERTENCIA: {count_dup_regional_todas} duplicados encontrados en regional todas")
    df_regional_todas_dup_check.show(10)
else:
    print("✓ No hay duplicados en agregación regional todas")

print("\n🔍 Verificando duplicados en df_pais_todas_final:")
df_pais_todas_dup_check = (
    df_pais_todas_final
    .groupBy("PAIS", "Familia", "Ano", "ORDENM", "PWOH")
    .agg(F.count("*").alias("count"))
    .filter(col("count") > 1)
)
count_dup_pais_todas = df_pais_todas_dup_check.count()
if count_dup_pais_todas > 0:
    print(f"⚠️  ADVERTENCIA: {count_dup_pais_todas} duplicados encontrados en país todas")
    df_pais_todas_dup_check.show(10)
else:
    print("✓ No hay duplicados en agregación país todas")

# ==============================================================================
# PASO 5: UNION Y GUARDADO
# ==============================================================================
print("\n[PASO 5] Uniendo todas las tablas y guardando...")

df_tabla_completa = (
    df_tabla_sin_duplicados
    .unionByName(df_regional_final)
    .unionByName(df_regional_todas_final)
    .unionByName(df_pais_todas_final)  # ← AGREGADO: Union con país todas
)

count_individual = df_tabla_sin_duplicados.count()
count_regional = df_regional_final.count()
count_regional_todas = df_regional_todas_final.count()
count_pais_todas = df_pais_todas_final.count()  # ← AGREGADO
count_total = df_tabla_completa.count()

print(f"✓ Individuales: {count_individual:,}")
print(f"✓ Regional por familia: {count_regional:,}")
print(f"✓ Regional TODAS familias: {count_regional_todas:,}")
print(f"✓ País TODAS familias: {count_pais_todas:,}")  # ← AGREGADO
print(f"✓ Total: {count_total:,}")

# Validación final de duplicados en tabla completa
print("\n🔍 Validación final de duplicados en tabla completa...")
df_tabla_completa_dup_check = (
    df_tabla_completa
    .groupBy("PAIS", "Familia", "Ano", "ORDENM", "PWOH")
    .agg(F.count("*").alias("count"))
    .filter(col("count") > 1)
)
count_dup_final = df_tabla_completa_dup_check.count()
if count_dup_final > 0:
    print(f"⚠️  ADVERTENCIA: {count_dup_final} duplicados encontrados en tabla completa")
    print("Mostrando ejemplos:")
    df_tabla_completa_dup_check.show(10)
else:
    print("✓ No hay duplicados en tabla completa")

df_tabla_final_opt = df_tabla_completa.repartition(200, "PAIS", "Familia", "PWOH")

spark.sql("USE LH_INTELLIGENT_VISIBILITY_CARATULAS")

try:
    spark.sql("DROP TABLE IF EXISTS tbl_caratula_cdv_clean")
    print("✓ Tabla anterior eliminada")
except Exception as e:
    print(f"⚠ Error: {e}")

df_tabla_final_opt.write.format("delta").mode("overwrite").saveAsTable("tbl_caratula_cdv_clean")

print("\n✅ TABLA GUARDADA: tbl_caratula_cdv_clean")
print(f"  • Total registros: {count_total:,}")

# ==============================================================================
# VERIFICACIÓN
# ==============================================================================
print("\n" + "="*80)
print("VERIFICACIÓN")
print("="*80)

print("\n📊 MUESTRA CEAM - Todas - PWOH 1:")
spark.sql("""
SELECT PAIS, Familia, ORDENM, Inv_Final, WHO, UND_TTO, WOH_TTO
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CEAM' AND Familia = 'Todas' AND Ano = 2025 AND PWOH = 1
ORDER BY ORDENM
""").show(12, truncate=False)

print("\n📊 MUESTRA México - Todas - PWOH 1 (NUEVO!):")
spark.sql("""
SELECT PAIS, Familia, ORDENM, Inv_Final, WHO, UND_TTO, WOH_TTO
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'MX' AND Familia = 'Todas' AND Ano = 2025 AND PWOH = 1
ORDER BY ORDENM
""").show(12, truncate=False)

print("\n📊 CONTEO POR FAMILIA:")
spark.sql("""
SELECT Familia, COUNT(*) as Registros
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
GROUP BY Familia
ORDER BY Familia
""").show(50, truncate=False)

print("\n📊 CONTEO POR PAIS Y FAMILIA (mostrando 'Todas'):")
spark.sql("""
SELECT PAIS, Familia, COUNT(*) as Registros
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE Familia = 'Todas'
GROUP BY PAIS, Familia
ORDER BY PAIS
""").show(50, truncate=False)

print("\n[✓] SNIPPET 6 COMPLETADO")

StatementMeta(, 5fc90911-6e7a-4e9b-8780-73c216b9965d, 8, Finished, Available, Finished)


SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL

[PASO 1] Cargando archivos de budget...
✓ BGT_PP cargado exitosamente: 32 registros
✓ BGT_CONT cargado exitosamente: 2,052 registros

[PASO 2] Calculando campos monetarios...
✓ Monetarios agregados: 68,770 registros

[PASO 3] Creando tabla final con estructura Power BI...
✓ Tabla final estructurada: 68,770 registros

[PASO 4] Eliminando duplicados por clave primaria...
✓ Registros antes: 68,770
✓ Registros después: 45,970
✓ Duplicados eliminados: 22,800

[PASO 4.5] Creando agregación regional por familia...
✓ Agregación regional por familia: 15,840 registros

[PASO 4.6] Creando agregación regional TODAS las familias...
✓ Agregación regional TODAS familias: 2,160 registros

[PASO 4.7] Creando agregación por PAÍS TODAS las familias...
✓ Agregación por PAÍS TODAS familias: 7,200 registros

[PASO 4.8] Validando que no existan duplicados en agregaciones...

🔍 Verificando duplicados en df_regional_final:
✓ No hay duplicados en agregación regio

In [ ]:
# ==============================================================================
# SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL (CON AGREGACIÓN REGIONAL)
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL")
print("="*80)

# ==============================================================================
# PASO 1: CARGAR ARCHIVOS BUDGET
# ==============================================================================
print("\n[PASO 1] Cargando archivos de budget...")

try:
    df_bgt_pp = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_PP.csv")
        .select(col("Pais").alias("PAIS"), col("Familia"), 
                col("PP").cast("double").alias("BGT_PP"))
    )
    print(f"✓ BGT_PP cargado exitosamente: {df_bgt_pp.count():,} registros")
except:
    print("⚠ BGT_PP no encontrado, usando defaults")
    df_bgt_pp = spark.createDataFrame([
        ("CR", "LAVA", 164.5), ("CO", "LAVA", 100.0), ("MX", "SAGA", 120.0)
    ], ["PAIS", "Familia", "BGT_PP"])

try:
    df_bgt_cont = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_CONT.csv")
        .select("PAIS", "Familia", "Attribute", col("ORDENM").cast("int"), 
                col("BGT_QTY").cast("double"), 
                col("BGT_$").cast("double").alias("BGT_USD"))
    )
    print(f"✓ BGT_CONT cargado exitosamente: {df_bgt_cont.count():,} registros")
except:
    print("⚠ BGT_CONT no encontrado, usando defaults")
    df_bgt_cont = spark.createDataFrame([
        ("CR", "LAVA", "ENE", 1, 2550.0, 100000.0),
        ("CR", "LAVA", "FEB", 2, 2219.0, 95000.0),
        ("CO", "LAVA", "ENE", 1, 1000.0, 80000.0),
        ("MX", "SAGA", "ENE", 1, 1500.0, 120000.0)
    ], ["PAIS", "Familia", "Attribute", "ORDENM", "BGT_QTY", "BGT_USD"])

# ==============================================================================
# PASO 2: CALCULAR MONETARIOS
# ==============================================================================
print("\n[PASO 2] Calculando campos monetarios...")

df_con_monetarios = (
    df_woh
    .join(df_bgt_pp, ["PAIS", "Familia"], "left")
    .join(df_bgt_cont, ["PAIS", "Familia", "Attribute", "ORDENM"], "left")
    .fillna(0, ["BGT_PP", "BGT_QTY", "BGT_USD"])
    
    # Columnas monetarias (en millones)
    .withColumn("Inv_Final_USD", (col("Inv_Final") * col("BGT_PP")) / 1_000_000)
    .withColumn("TTOS_USD", (col("TTOS") * col("BGT_PP")) / 1_000_000)
    .withColumn("USD_OH_TTOS", col("Inv_Final_USD") + col("TTOS_USD"))
    
    # Diferencias vs budget
    .withColumn("Dif_Real_vs_BGT_QTY", col("UND_TTOS") - col("BGT_QTY"))
    .withColumn("Dif_Real_vs_BGT_USD", col("USD_OH_TTOS") - col("BGT_USD"))
)

print(f"✓ Monetarios agregados: {df_con_monetarios.count():,} registros")

# ==============================================================================
# PASO 3: CREAR TABLA FINAL PARA POWER BI
# ==============================================================================
print("\n[PASO 3] Creando tabla final con estructura Power BI...")

df_tabla_final = (
    df_con_monetarios
    .select(
        col("Ano").cast("int"),
        col("Mes"),
        when(col("PAIS").isin("CO", "EC", "PE", "AR", "CL"), "Andino")
        .when(col("PAIS").isin("CR", "GT", "HN", "SV", "MX"), "CEAM")
        .when(col("PAIS").isin("CA"), "NorteAmerica")
        .otherwise("Otros").alias("Region"),
        col("PAIS"),
        col("Familia"),
        col("Attribute"),
        col("Inv_Inic").cast("double"),
        col("Llegadas").cast("double"),
        col("Fcst").alias("S&PO").cast("double"),
        col("Facturacion").alias("Facturado").cast("double"),
        col("Inv_Final").cast("double"),
        F.round(col("Inv_Final_USD"), 4).alias("Inv_Final_$"),
        col("WOH").alias("WHO").cast("int"),
        col("TTOS").cast("double"),
        F.round(col("TTOS_USD"), 4).alias("TTOS_$"),
        F.round(col("USD_OH_TTOS"), 4).alias("USD_OH_TTOS"),
        col("UND_TTOS").alias("UND_TTO").cast("double"),
        col("WOH_TTOS").alias("WOH_TTO").cast("int"),
        col("BGT_QTY").cast("double"),
        col("BGT_USD").alias("BGT_$").cast("double"),
        col("Dif_Real_vs_BGT_QTY").cast("double"),
        col("Dif_Real_vs_BGT_USD").alias("Dif_Real_vs_BGT_$").cast("double"),
        col("PWOH").cast("double"),
        col("ORDENM").cast("double"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Tabla final estructurada: {df_tabla_final.count():,} registros")

# ==============================================================================
# PASO 4: ELIMINAR DUPLICADOS
# ==============================================================================
print("\n[PASO 4] Eliminando duplicados por clave primaria...")

count_antes = df_tabla_final.count()

df_tabla_sin_duplicados = df_tabla_final.dropDuplicates([
    "PAIS", "Familia", "Ano", "ORDENM", "PWOH"
])

count_despues = df_tabla_sin_duplicados.count()
duplicados_eliminados = count_antes - count_despues

print(f"✓ Registros antes: {count_antes:,}")
print(f"✓ Registros después: {count_despues:,}")
print(f"✓ Duplicados eliminados: {duplicados_eliminados:,}")

# ==============================================================================
# PASO 4.5: AGREGACIÓN REGIONAL POR FAMILIA
# ==============================================================================
print("\n[PASO 4.5] Creando agregación regional por familia...")

df_regional = (
    df_tabla_sin_duplicados
    .groupBy("Region", "Familia", "Ano", "Mes", "Attribute", "ORDENM", "PWOH")
    .agg(
        F.sum("Inv_Inic").alias("Inv_Inic"),
        F.sum("Llegadas").alias("Llegadas"),
        F.sum("S&PO").alias("S&PO"),
        F.sum("Facturado").alias("Facturado"),
        F.sum("Inv_Final").alias("Inv_Final"),
        F.sum("Inv_Final_$").alias("Inv_Final_$"),
        F.sum("TTOS").alias("TTOS"),
        F.sum("TTOS_$").alias("TTOS_$"),
        F.sum("USD_OH_TTOS").alias("USD_OH_TTOS"),
        F.sum("UND_TTO").alias("UND_TTO"),
        F.sum("BGT_QTY").alias("BGT_QTY"),
        F.sum("BGT_$").alias("BGT_$"),
        F.sum("Dif_Real_vs_BGT_QTY").alias("Dif_Real_vs_BGT_QTY"),
        F.sum("Dif_Real_vs_BGT_$").alias("Dif_Real_vs_BGT_$")
    )
)

w_reg = Window.partitionBy("Region", "Familia", "PWOH").orderBy("Ano", "ORDENM")

df_regional = (
    df_regional
    .withColumn("fcst_m1", lead("S&PO", 1).over(w_reg))
    .withColumn("fcst_m2", lead("S&PO", 2).over(w_reg))
)

df_ult_fact_reg = (
    df_regional
    .filter(col("Facturado") > 0)
    .groupBy("Region", "Familia", "Ano", "PWOH")
    .agg(F.max("ORDENM").alias("ult_fact"))
)

df_regional = (
    df_regional
    .join(df_ult_fact_reg, ["Region", "Familia", "Ano", "PWOH"], "left")
    .withColumn("ult_fact", coalesce(col("ult_fact"), lit(0)))
    .withColumn(
        "denom",
        when(col("ORDENM") <= col("ult_fact"),
             (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0)
        .otherwise(coalesce(col("fcst_m1"), lit(0.0)) / 4.0)
    )
    .withColumn("WHO", 
                when(col("denom") > 0, F.round(col("Inv_Final") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .withColumn("WOH_TTO", 
                when(col("denom") > 0, F.round(col("UND_TTO") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .drop("fcst_m1", "fcst_m2", "denom", "ult_fact")
)

df_regional_final = (
    df_regional
    .select(
        col("Ano"), col("Mes"), col("Region"),
        col("Region").alias("PAIS"),
        col("Familia"),
        col("Attribute"), col("Inv_Inic"), col("Llegadas"), col("S&PO"), col("Facturado"),
        col("Inv_Final"), col("Inv_Final_$"), col("WHO"), col("TTOS"), col("TTOS_$"),
        col("USD_OH_TTOS"), col("UND_TTO"), col("WOH_TTO"), col("BGT_QTY"), col("BGT_$"),
        col("Dif_Real_vs_BGT_QTY"), col("Dif_Real_vs_BGT_$"), col("PWOH"), col("ORDENM"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Agregación regional por familia: {df_regional_final.count():,} registros")

# ==============================================================================
# PASO 4.6: AGREGACIÓN REGIONAL TODAS LAS FAMILIAS
# ==============================================================================
print("\n[PASO 4.6] Creando agregación regional TODAS las familias...")

df_regional_todas = (
    df_tabla_sin_duplicados
    .groupBy("Region", "Ano", "Mes", "Attribute", "ORDENM", "PWOH")  # SIN Familia
    .agg(
        F.sum("Inv_Inic").alias("Inv_Inic"),
        F.sum("Llegadas").alias("Llegadas"),
        F.sum("S&PO").alias("S&PO"),
        F.sum("Facturado").alias("Facturado"),
        F.sum("Inv_Final").alias("Inv_Final"),
        F.sum("Inv_Final_$").alias("Inv_Final_$"),
        F.sum("TTOS").alias("TTOS"),
        F.sum("TTOS_$").alias("TTOS_$"),
        F.sum("USD_OH_TTOS").alias("USD_OH_TTOS"),
        F.sum("UND_TTO").alias("UND_TTO"),
        F.sum("BGT_QTY").alias("BGT_QTY"),
        F.sum("BGT_$").alias("BGT_$"),
        F.sum("Dif_Real_vs_BGT_QTY").alias("Dif_Real_vs_BGT_QTY"),
        F.sum("Dif_Real_vs_BGT_$").alias("Dif_Real_vs_BGT_$")
    )
)

w_reg_todas = Window.partitionBy("Region", "PWOH").orderBy("Ano", "ORDENM")

df_regional_todas = (
    df_regional_todas
    .withColumn("fcst_m1", lead("S&PO", 1).over(w_reg_todas))
    .withColumn("fcst_m2", lead("S&PO", 2).over(w_reg_todas))
)

df_ult_fact_reg_todas = (
    df_regional_todas
    .filter(col("Facturado") > 0)
    .groupBy("Region", "Ano", "PWOH")
    .agg(F.max("ORDENM").alias("ult_fact"))
)

df_regional_todas = (
    df_regional_todas
    .join(df_ult_fact_reg_todas, ["Region", "Ano", "PWOH"], "left")
    .withColumn("ult_fact", coalesce(col("ult_fact"), lit(0)))
    .withColumn(
        "denom",
        when(col("ORDENM") <= col("ult_fact"),
             (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0)
        .otherwise(coalesce(col("fcst_m1"), lit(0.0)) / 4.0)
    )
    .withColumn("WHO", 
                when(col("denom") > 0, F.round(col("Inv_Final") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .withColumn("WOH_TTO", 
                when(col("denom") > 0, F.round(col("UND_TTO") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .drop("fcst_m1", "fcst_m2", "denom", "ult_fact")
)

df_regional_todas_final = (
    df_regional_todas
    .select(
        col("Ano"), col("Mes"), col("Region"),
        col("Region").alias("PAIS"),
        lit("Todas").alias("Familia"),  # ← FAMILIA = "Todas"
        col("Attribute"), col("Inv_Inic"), col("Llegadas"), col("S&PO"), col("Facturado"),
        col("Inv_Final"), col("Inv_Final_$"), col("WHO"), col("TTOS"), col("TTOS_$"),
        col("USD_OH_TTOS"), col("UND_TTO"), col("WOH_TTO"), col("BGT_QTY"), col("BGT_$"),
        col("Dif_Real_vs_BGT_QTY"), col("Dif_Real_vs_BGT_$"), col("PWOH"), col("ORDENM"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Agregación regional TODAS familias: {df_regional_todas_final.count():,} registros")

# ==============================================================================
# PASO 5: UNION Y GUARDADO
# ==============================================================================
print("\n[PASO 5] Uniendo todas las tablas y guardando...")

df_tabla_completa = (
    df_tabla_sin_duplicados
    .unionByName(df_regional_final)
    .unionByName(df_regional_todas_final)
)

count_individual = df_tabla_sin_duplicados.count()
count_regional = df_regional_final.count()
count_regional_todas = df_regional_todas_final.count()
count_total = df_tabla_completa.count()

print(f"✓ Individuales: {count_individual:,}")
print(f"✓ Regional por familia: {count_regional:,}")
print(f"✓ Regional TODAS familias: {count_regional_todas:,}")
print(f"✓ Total: {count_total:,}")

df_tabla_final_opt = df_tabla_completa.repartition(200, "PAIS", "Familia", "PWOH")

spark.sql("USE LH_INTELLIGENT_VISIBILITY_CARATULAS")

try:
    spark.sql("DROP TABLE IF EXISTS tbl_caratula_cdv_clean")
    print("✓ Tabla anterior eliminada")
except Exception as e:
    print(f"⚠ Error: {e}")

df_tabla_final_opt.write.format("delta").mode("overwrite").saveAsTable("tbl_caratula_cdv_clean")

print("\n✅ TABLA GUARDADA: tbl_caratula_cdv_clean")
print(f"  • Total registros: {count_total:,}")

# ==============================================================================
# VERIFICACIÓN
# ==============================================================================
print("\n" + "="*80)
print("VERIFICACIÓN")
print("="*80)

print("\n📊 MUESTRA CEAM - Todas - PWOH 1:")
spark.sql("""
SELECT PAIS, Familia, ORDENM, Inv_Final, WHO, UND_TTO, WOH_TTO
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CEAM' AND Familia = 'Todas' AND Ano = 2025 AND PWOH = 1
ORDER BY ORDENM
""").show(12, truncate=False)

print("\n📊 CONTEO POR FAMILIA:")
spark.sql("""
SELECT Familia, COUNT(*) as Registros
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
GROUP BY Familia
ORDER BY Familia
""").show(50, truncate=False)

print("\n[✓] SNIPPET 6 COMPLETADO")

In [6]:
# ==============================================================================
# SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL (CON AGREGACIÓN REGIONAL)
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL")
print("="*80)

# ==============================================================================
# PASO 1: CARGAR ARCHIVOS BUDGET
# ==============================================================================
print("\n[PASO 1] Cargando archivos de budget...")

try:
    df_bgt_pp = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_PP.csv")
        .select(col("Pais").alias("PAIS"), col("Familia"), 
                col("PP").cast("double").alias("BGT_PP"))
    )
    print(f"✓ BGT_PP cargado exitosamente: {df_bgt_pp.count():,} registros")
except:
    print("⚠ BGT_PP no encontrado, usando defaults")
    df_bgt_pp = spark.createDataFrame([
        ("CR", "LAVA", 164.5), ("CO", "LAVA", 100.0), ("MX", "SAGA", 120.0)
    ], ["PAIS", "Familia", "BGT_PP"])

try:
    df_bgt_cont = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_CONT.csv")
        .select("PAIS", "Familia", "Attribute", col("ORDENM").cast("int"), 
                col("BGT_QTY").cast("double"), 
                col("BGT_$").cast("double").alias("BGT_USD"))
    )
    print(f"✓ BGT_CONT cargado exitosamente: {df_bgt_cont.count():,} registros")
except:
    print("⚠ BGT_CONT no encontrado, usando defaults")
    df_bgt_cont = spark.createDataFrame([
        ("CR", "LAVA", "ENE", 1, 2550.0, 100000.0),
        ("CR", "LAVA", "FEB", 2, 2219.0, 95000.0),
        ("CO", "LAVA", "ENE", 1, 1000.0, 80000.0),
        ("MX", "SAGA", "ENE", 1, 1500.0, 120000.0)
    ], ["PAIS", "Familia", "Attribute", "ORDENM", "BGT_QTY", "BGT_USD"])

# ==============================================================================
# PASO 2: CALCULAR MONETARIOS
# ==============================================================================
print("\n[PASO 2] Calculando campos monetarios...")

df_con_monetarios = (
    df_woh
    .join(df_bgt_pp, ["PAIS", "Familia"], "left")
    .join(df_bgt_cont, ["PAIS", "Familia", "Attribute", "ORDENM"], "left")
    .fillna(0, ["BGT_PP", "BGT_QTY", "BGT_USD"])
    
    # Columnas monetarias (en millones)
    .withColumn("Inv_Final_USD", (col("Inv_Final") * col("BGT_PP")) / 1_000_000)
    .withColumn("TTOS_USD", (col("TTOS") * col("BGT_PP")) / 1_000_000)
    .withColumn("USD_OH_TTOS", col("Inv_Final_USD") + col("TTOS_USD"))
    
    # Diferencias vs budget
    .withColumn("Dif_Real_vs_BGT_QTY", col("UND_TTOS") - col("BGT_QTY"))
    .withColumn("Dif_Real_vs_BGT_USD", col("USD_OH_TTOS") - col("BGT_USD"))
)

print(f"✓ Monetarios agregados: {df_con_monetarios.count():,} registros")

# ==============================================================================
# PASO 3: CREAR TABLA FINAL PARA POWER BI
# ==============================================================================
print("\n[PASO 3] Creando tabla final con estructura Power BI...")

df_tabla_final = (
    df_con_monetarios
    .select(
        col("Ano").cast("int"),
        col("Mes"),
        when(col("PAIS").isin("CO", "EC", "PE", "AR", "CL"), "Andino")
        .when(col("PAIS").isin("CR", "GT", "HN", "SV", "MX"), "CEAM")
        .when(col("PAIS").isin("CA"), "NorteAmerica")
        .otherwise("Otros").alias("Region"),
        col("PAIS"),
        col("Familia"),
        col("Attribute"),
        col("Inv_Inic").cast("double"),
        col("Llegadas").cast("double"),
        col("Fcst").alias("S&PO").cast("double"),
        col("Facturacion").alias("Facturado").cast("double"),
        col("Inv_Final").cast("double"),
        F.round(col("Inv_Final_USD"), 4).alias("Inv_Final_$"),
        col("WOH").alias("WHO").cast("int"),
        col("TTOS").cast("double"),
        F.round(col("TTOS_USD"), 4).alias("TTOS_$"),
        F.round(col("USD_OH_TTOS"), 4).alias("USD_OH_TTOS"),
        col("UND_TTOS").alias("UND_TTO").cast("double"),
        col("WOH_TTOS").alias("WOH_TTO").cast("int"),
        col("BGT_QTY").cast("double"),
        col("BGT_USD").alias("BGT_$").cast("double"),
        col("Dif_Real_vs_BGT_QTY").cast("double"),
        col("Dif_Real_vs_BGT_USD").alias("Dif_Real_vs_BGT_$").cast("double"),
        col("PWOH").cast("double"),
        col("ORDENM").cast("double"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Tabla final estructurada: {df_tabla_final.count():,} registros")

# ==============================================================================
# PASO 4: ELIMINAR DUPLICADOS
# ==============================================================================
print("\n[PASO 4] Eliminando duplicados por clave primaria...")

count_antes = df_tabla_final.count()

df_tabla_sin_duplicados = df_tabla_final.dropDuplicates([
    "PAIS", "Familia", "Ano", "ORDENM", "PWOH"
])

count_despues = df_tabla_sin_duplicados.count()
duplicados_eliminados = count_antes - count_despues

print(f"✓ Registros antes: {count_antes:,}")
print(f"✓ Registros después: {count_despues:,}")
print(f"✓ Duplicados eliminados: {duplicados_eliminados:,}")

# ==============================================================================
# PASO 4.5: AGREGACIÓN REGIONAL
# ==============================================================================
print("\n[PASO 4.5] Creando agregación regional...")

# Agrupar y sumar por región
df_regional = (
    df_tabla_sin_duplicados
    .groupBy("Region", "Familia", "Ano", "Mes", "Attribute", "ORDENM", "PWOH")
    .agg(
        F.sum("Inv_Inic").alias("Inv_Inic"),
        F.sum("Llegadas").alias("Llegadas"),
        F.sum("S&PO").alias("S&PO"),
        F.sum("Facturado").alias("Facturado"),
        F.sum("Inv_Final").alias("Inv_Final"),
        F.sum("Inv_Final_$").alias("Inv_Final_$"),
        F.sum("TTOS").alias("TTOS"),
        F.sum("TTOS_$").alias("TTOS_$"),
        F.sum("USD_OH_TTOS").alias("USD_OH_TTOS"),
        F.sum("UND_TTO").alias("UND_TTO"),
        F.sum("BGT_QTY").alias("BGT_QTY"),
        F.sum("BGT_$").alias("BGT_$"),
        F.sum("Dif_Real_vs_BGT_QTY").alias("Dif_Real_vs_BGT_QTY"),
        F.sum("Dif_Real_vs_BGT_$").alias("Dif_Real_vs_BGT_$")
    )
)

print(f"✓ Agregación regional creada: {df_regional.count():,} registros")

# Window para calcular forecasts futuros
w_reg = Window.partitionBy("Region", "Familia", "PWOH").orderBy("Ano", "ORDENM")

df_regional = (
    df_regional
    .withColumn("fcst_m1", lead("S&PO", 1).over(w_reg))
    .withColumn("fcst_m2", lead("S&PO", 2).over(w_reg))
)

# Detectar último mes facturado regional
df_ult_fact_reg = (
    df_regional
    .filter(col("Facturado") > 0)
    .groupBy("Region", "Familia", "Ano", "PWOH")
    .agg(F.max("ORDENM").alias("ult_fact"))
)

# Calcular WHO y WOH_TTO regional
df_regional = (
    df_regional
    .join(df_ult_fact_reg, ["Region", "Familia", "Ano", "PWOH"], "left")
    .withColumn("ult_fact", coalesce(col("ult_fact"), lit(0)))
    .withColumn(
        "denom",
        when(col("ORDENM") <= col("ult_fact"),
             (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0)
        .otherwise(coalesce(col("fcst_m1"), lit(0.0)) / 4.0)
    )
    .withColumn("WHO", 
                when(col("denom") > 0, F.round(col("Inv_Final") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .withColumn("WOH_TTO", 
                when(col("denom") > 0, F.round(col("UND_TTO") / col("denom"), 0).cast("int"))
                .otherwise(lit(0)))
    .drop("fcst_m1", "fcst_m2", "denom", "ult_fact")
)

print(f"✓ WHO y WOH_TTO regional calculados")

# Dar formato final con PAIS = Region
df_regional_final = (
    df_regional
    .select(
        col("Ano"),
        col("Mes"),
        col("Region"),
        col("Region").alias("PAIS"),           # ✅ PAIS = Region (Andino, CEAM, etc.)
        col("Familia"),
        col("Attribute"),
        col("Inv_Inic"),
        col("Llegadas"),
        col("S&PO"),
        col("Facturado"),
        col("Inv_Final"),
        col("Inv_Final_$"),
        col("WHO"),
        col("TTOS"),
        col("TTOS_$"),
        col("USD_OH_TTOS"),
        col("UND_TTO"),
        col("WOH_TTO"),
        col("BGT_QTY"),
        col("BGT_$"),
        col("Dif_Real_vs_BGT_QTY"),
        col("Dif_Real_vs_BGT_$"),
        col("PWOH"),
        col("ORDENM"),
        lit("Cubos").alias("Calculo")          # ✅ Mismo valor que individuales
    )
)

print(f"✓ Tabla regional formateada: {df_regional_final.count():,} registros")

# Union de datos individuales + regionales
df_tabla_completa = df_tabla_sin_duplicados.unionByName(df_regional_final)

count_individual = df_tabla_sin_duplicados.count()
count_regional = df_regional_final.count()
count_total = df_tabla_completa.count()

print(f"✓ Total individual: {count_individual:,}")
print(f"✓ Total regional: {count_regional:,}")
print(f"✓ Total completo: {count_total:,}")

# ==============================================================================
# PASO 5: OPTIMIZAR Y GUARDAR
# ==============================================================================
print("\n[PASO 5] Optimizando y guardando tabla final completa...")

# Optimización final con particionamiento
df_tabla_final_opt = df_tabla_completa.repartition(200, "PAIS", "Familia", "PWOH")

# Usar esquema
spark.sql("USE LH_INTELLIGENT_VISIBILITY_CARATULAS")

# Eliminar tabla anterior si existe
try:
    spark.sql("DROP TABLE IF EXISTS tbl_caratula_cdv_clean")
    print("✓ Tabla anterior eliminada")
except Exception as e:
    print(f"⚠ No se pudo eliminar tabla anterior: {e}")

# Guardar nueva tabla
df_tabla_final_opt.write.format("delta").mode("overwrite").saveAsTable("tbl_caratula_cdv_clean")

print("\n✅ TABLA FINAL GUARDADA:")
print("  • Nombre: tbl_caratula_cdv_clean")
print("  • Esquema: LH_INTELLIGENT_VISIBILITY_CARATULAS")
print(f"  • Registros individuales: {count_individual:,}")
print(f"  • Registros regionales: {count_regional:,}")
print(f"  • Total registros: {count_total:,}")
print(f"  • Particiones: 200")

# ==============================================================================
# VERIFICACIÓN FINAL
# ==============================================================================
print("\n" + "="*80)
print("VERIFICACIÓN FINAL")
print("="*80)

print("\n📊 SAMPLE CR-LAVA PWOH 1 - 2025 (Individual):")
spark.sql("""
SELECT PAIS, Region, Attribute, ORDENM, Inv_Inic, Llegadas, `S&PO`, Facturado, 
       Inv_Final, WHO, TTOS, UND_TTO, Calculo
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CR' AND Familia = 'LAVA' AND Ano = 2025 AND PWOH = 1
ORDER BY ORDENM
""").show(12, truncate=False)

print("\n📊 SAMPLE REGIONAL - Andino LAVA PWOH 1 - 2025 (Agregado):")
spark.sql("""
SELECT PAIS, Region, Attribute, ORDENM, Inv_Inic, Llegadas, `S&PO`, Facturado, 
       Inv_Final, WHO, TTOS, UND_TTO, Calculo
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'Andino' AND Familia = 'LAVA' AND Ano = 2025 AND PWOH = 1
ORDER BY ORDENM
""").show(12, truncate=False)

print("\n📊 COMPARACIÓN: Individual vs Regional (ENERO 2025, PWOH=1):")
spark.sql("""
SELECT PAIS, Region, Inv_Final, WHO, UND_TTO, WOH_TTO, Calculo
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE Region = 'Andino' AND Familia = 'LAVA' 
  AND Ano = 2025 AND ORDENM = 1 AND PWOH = 1
ORDER BY PAIS
""").show(20, truncate=False)

print("\n📊 VALORES ÚNICOS DE PAIS (debe incluir países + regiones):")
spark.sql("""
SELECT DISTINCT PAIS, Region
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
ORDER BY Region, PAIS
""").show(50, truncate=False)

print("\n📊 ESTADÍSTICAS GENERALES:")
spark.sql("""
SELECT 
    COUNT(*) as total_registros,
    COUNT(DISTINCT PAIS) as paises_y_regiones,
    COUNT(DISTINCT Region) as regiones,
    COUNT(DISTINCT Familia) as familias,
    COUNT(DISTINCT Ano) as anos,
    COUNT(DISTINCT PWOH) as pwoh_values
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
""").show()

print("\n📊 CONTEO POR PAIS (Individuales vs Agregados):")
spark.sql("""
SELECT 
    PAIS,
    Region,
    COUNT(*) as Registros,
    COUNT(DISTINCT Familia) as Familias
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
GROUP BY PAIS, Region
ORDER BY Region, PAIS
""").show(50, truncate=False)

print("\n📊 VERIFICACIÓN DE INTEGRIDAD:")
spark.sql("""
SELECT 
    PAIS,
    Familia,
    Ano,
    COUNT(*) as registros_por_ano,
    COUNT(DISTINCT ORDENM) as meses_distintos,
    COUNT(DISTINCT PWOH) as pwoh_distintos
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
GROUP BY PAIS, Familia, Ano
ORDER BY PAIS, Familia, Ano
""").show(50, truncate=False)

print("\n" + "="*80)
print("✅ PROCESO COMPLETADO - TABLA LISTA PARA POWER BI")
print("="*80)
print("RESUMEN:")
print("✓ Ancla global detectado correctamente")
print("✓ PWOH 1-10 expandidos en TODOS los meses")
print("✓ Llegadas calculadas con fórmula correcta donde son 0")
print("✓ Inventarios propagados mes a mes por PWOH")
print("✓ TTOS calculado con llegadas futuras")
print("✓ WOH calculado con fórmulas complejas")
print("✓ Monetarios y budget agregados")
print("✓ Duplicados eliminados")
print("✓ Agregación regional: PAIS=Region (Andino, CEAM, NorteAmerica)")
print("✓ Todos con Calculo='Cubos'")
print("✓ Tabla final guardada: tbl_caratula_cdv_clean")
print("="*80)

print("\n[✓] SNIPPET 6 COMPLETADO")

StatementMeta(, 76a626dd-3496-481b-8724-e72c3d43de18, 8, Finished, Available, Finished)


SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL

[PASO 1] Cargando archivos de budget...
✓ BGT_PP cargado exitosamente: 32 registros
✓ BGT_CONT cargado exitosamente: 2,052 registros

[PASO 2] Calculando campos monetarios...
✓ Monetarios agregados: 68,770 registros

[PASO 3] Creando tabla final con estructura Power BI...
✓ Tabla final estructurada: 68,770 registros

[PASO 4] Eliminando duplicados por clave primaria...
✓ Registros antes: 68,770
✓ Registros después: 45,970
✓ Duplicados eliminados: 22,800

[PASO 4.5] Creando agregación regional...
✓ Agregación regional creada: 15,840 registros
✓ WHO y WOH_TTO regional calculados
✓ Tabla regional formateada: 15,840 registros
✓ Total individual: 45,970
✓ Total regional: 15,840
✓ Total completo: 61,810

[PASO 5] Optimizando y guardando tabla final completa...
✓ Tabla anterior eliminada

✅ TABLA FINAL GUARDADA:
  • Nombre: tbl_caratula_cdv_clean
  • Esquema: LH_INTELLIGENT_VISIBILITY_CARATULAS
  • Registros individuales: 45,970
  • Registros r

In [6]:
# ==============================================================================
# SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL")
print("="*80)

# ==============================================================================
# PASO 1: CARGAR ARCHIVOS BUDGET
# ==============================================================================
print("\n[PASO 1] Cargando archivos de budget...")

try:
    df_bgt_pp = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_PP.csv")
        .select(col("Pais").alias("PAIS"), col("Familia"), 
                col("PP").cast("double").alias("BGT_PP"))
    )
    print(f"✓ BGT_PP cargado exitosamente: {df_bgt_pp.count():,} registros")
except:
    print("⚠ BGT_PP no encontrado, usando defaults")
    df_bgt_pp = spark.createDataFrame([
        ("CR", "LAVA", 164.5), ("CO", "LAVA", 100.0), ("MX", "SAGA", 120.0)
    ], ["PAIS", "Familia", "BGT_PP"])

try:
    df_bgt_cont = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_CONT.csv")
        .select("PAIS", "Familia", "Attribute", col("ORDENM").cast("int"), 
                col("BGT_QTY").cast("double"), 
                col("BGT_$").cast("double").alias("BGT_USD"))
    )
    print(f"✓ BGT_CONT cargado exitosamente: {df_bgt_cont.count():,} registros")
except:
    print("⚠ BGT_CONT no encontrado, usando defaults")
    df_bgt_cont = spark.createDataFrame([
        ("CR", "LAVA", "ENE", 1, 2550.0, 100000.0),
        ("CR", "LAVA", "FEB", 2, 2219.0, 95000.0),
        ("CO", "LAVA", "ENE", 1, 1000.0, 80000.0),
        ("MX", "SAGA", "ENE", 1, 1500.0, 120000.0)
    ], ["PAIS", "Familia", "Attribute", "ORDENM", "BGT_QTY", "BGT_USD"])

# ==============================================================================
# PASO 2: CALCULAR MONETARIOS
# ==============================================================================
print("\n[PASO 2] Calculando campos monetarios...")

df_con_monetarios = (
    df_woh
    .join(df_bgt_pp, ["PAIS", "Familia"], "left")
    .join(df_bgt_cont, ["PAIS", "Familia", "Attribute", "ORDENM"], "left")
    .fillna(0, ["BGT_PP", "BGT_QTY", "BGT_USD"])
    
    # Columnas monetarias (en millones)
    .withColumn("Inv_Final_USD", (col("Inv_Final") * col("BGT_PP")) / 1_000_000)
    .withColumn("TTOS_USD", (col("TTOS") * col("BGT_PP")) / 1_000_000)
    .withColumn("USD_OH_TTOS", col("Inv_Final_USD") + col("TTOS_USD"))
    
    # Diferencias vs budget
    .withColumn("Dif_Real_vs_BGT_QTY", col("UND_TTOS") - col("BGT_QTY"))
    .withColumn("Dif_Real_vs_BGT_USD", col("USD_OH_TTOS") - col("BGT_USD"))
)

print(f"✓ Monetarios agregados: {df_con_monetarios.count():,} registros")

# ==============================================================================
# PASO 3: CREAR TABLA FINAL PARA POWER BI
# ==============================================================================
print("\n[PASO 3] Creando tabla final con estructura Power BI...")

df_tabla_final = (
    df_con_monetarios
    .select(
        col("Ano").cast("int"),
        col("Mes"),
        when(col("PAIS").isin("CO", "EC", "PE", "AR", "CL"), "Andino")
        .when(col("PAIS").isin("CR", "GT", "HN", "SV", "MX"), "CEAM")
        .when(col("PAIS").isin("CA"), "NorteAmerica")
        .otherwise("Otros").alias("Region"),
        col("PAIS"),
        col("Familia"),
        col("Attribute"),
        col("Inv_Inic").cast("double"),
        col("Llegadas").cast("double"),
        col("Fcst").alias("S&PO").cast("double"),
        col("Facturacion").alias("Facturado").cast("double"),
        col("Inv_Final").cast("double"),
        F.round(col("Inv_Final_USD"), 4).alias("Inv_Final_$"),
        col("WOH").alias("WHO").cast("int"),
        col("TTOS").cast("double"),
        F.round(col("TTOS_USD"), 4).alias("TTOS_$"),
        F.round(col("USD_OH_TTOS"), 4).alias("USD_OH_TTOS"),
        col("UND_TTOS").alias("UND_TTO").cast("double"),
        col("WOH_TTOS").alias("WOH_TTO").cast("int"),
        col("BGT_QTY").cast("double"),
        col("BGT_USD").alias("BGT_$").cast("double"),
        col("Dif_Real_vs_BGT_QTY").cast("double"),
        col("Dif_Real_vs_BGT_USD").alias("Dif_Real_vs_BGT_$").cast("double"),
        col("PWOH").cast("double"),
        col("ORDENM").cast("double"),
        lit("Cubos").alias("Calculo")
    )
)

print(f"✓ Tabla final estructurada: {df_tabla_final.count():,} registros")

# ==============================================================================
# PASO 4: ELIMINAR DUPLICADOS
# ==============================================================================
print("\n[PASO 4] Eliminando duplicados por clave primaria...")

count_antes = df_tabla_final.count()

df_tabla_sin_duplicados = df_tabla_final.dropDuplicates([
    "PAIS", "Familia", "Ano", "ORDENM", "PWOH"
])

count_despues = df_tabla_sin_duplicados.count()
duplicados_eliminados = count_antes - count_despues

print(f"✓ Registros antes: {count_antes:,}")
print(f"✓ Registros después: {count_despues:,}")
print(f"✓ Duplicados eliminados: {duplicados_eliminados:,}")

# ==============================================================================
# PASO 5: OPTIMIZAR Y GUARDAR
# ==============================================================================
print("\n[PASO 5] Optimizando y guardando tabla final...")

# Optimización final con particionamiento
df_tabla_final_opt = df_tabla_sin_duplicados.repartition(200, "PAIS", "Familia", "PWOH")

# Usar esquema
spark.sql("USE LH_INTELLIGENT_VISIBILITY_CARATULAS")

# Eliminar tabla anterior si existe
try:
    spark.sql("DROP TABLE IF EXISTS tbl_caratula_cdv_clean")
    print("✓ Tabla anterior eliminada")
except Exception as e:
    print(f"⚠ No se pudo eliminar tabla anterior: {e}")

# Guardar nueva tabla
df_tabla_final_opt.write.format("delta").mode("overwrite").saveAsTable("tbl_caratula_cdv_clean")

print("\n✅ TABLA FINAL GUARDADA:")
print("  • Nombre: tbl_caratula_cdv_clean")
print("  • Esquema: LH_INTELLIGENT_VISIBILITY_CARATULAS")
print(f"  • Registros finales: {count_despues:,}")
print(f"  • Particiones: 200")

# ==============================================================================
# VERIFICACIÓN FINAL
# ==============================================================================
print("\n" + "="*80)
print("VERIFICACIÓN FINAL")
print("="*80)

print("\n📊 SAMPLE CR-LAVA PWOH 1 - 2025:")
spark.sql("""
SELECT Attribute, ORDENM, Inv_Inic, Llegadas, `S&PO`, Facturado, 
       Inv_Final, WHO, TTOS, UND_TTO, `Inv_Final_$`, `TTOS_$`, USD_OH_TTOS
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CR' AND Familia = 'LAVA' AND Ano = 2025 AND PWOH = 1
ORDER BY ORDENM
""").show(12, truncate=False)

print("\n📊 SAMPLE CR-LAVA PWOH 5 - 2025:")
spark.sql("""
SELECT Attribute, ORDENM, Inv_Inic, Llegadas, `S&PO`, Facturado, 
       Inv_Final, WHO, TTOS, UND_TTO
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
WHERE PAIS = 'CR' AND Familia = 'LAVA' AND Ano = 2025 AND PWOH = 5
ORDER BY ORDENM
""").show(12, truncate=False)

print("\n📊 ESTADÍSTICAS GENERALES:")
spark.sql("""
SELECT 
    COUNT(*) as total_registros,
    COUNT(DISTINCT PAIS) as paises,
    COUNT(DISTINCT Familia) as familias,
    COUNT(DISTINCT Ano) as anos,
    COUNT(DISTINCT PWOH) as pwoh_values
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
""").show()

print("\n📊 VERIFICACIÓN DE INTEGRIDAD:")
spark.sql("""
SELECT 
    PAIS,
    Familia,
    Ano,
    COUNT(*) as registros_por_ano,
    COUNT(DISTINCT ORDENM) as meses_distintos,
    COUNT(DISTINCT PWOH) as pwoh_distintos
FROM LH_INTELLIGENT_VISIBILITY_CARATULAS.tbl_caratula_cdv_clean
GROUP BY PAIS, Familia, Ano
ORDER BY PAIS, Familia, Ano
""").show(50, truncate=False)

print("\n" + "="*80)
print("✅ PROCESO COMPLETADO - TABLA LISTA PARA POWER BI")
print("="*80)
print("RESUMEN:")
print("✓ Ancla global detectado correctamente")
print("✓ PWOH 1-10 expandidos en TODOS los meses")
print("✓ Llegadas calculadas con fórmula correcta donde son 0")
print("✓ Inventarios propagados mes a mes por PWOH")
print("✓ TTOS calculado con llegadas futuras")
print("✓ WOH calculado con fórmulas complejas")
print("✓ Monetarios y budget agregados")
print("✓ Duplicados eliminados")
print("✓ Tabla final guardada: tbl_caratula_cdv_clean")
print("="*80)

print("\n[✓] SNIPPET 6 COMPLETADO")

StatementMeta(, b44d413b-4cca-4cd8-b43c-d4ddab87e1e7, 8, Finished, Available, Finished)


SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL

[PASO 1] Cargando archivos de budget...
✓ BGT_PP cargado exitosamente: 32 registros
✓ BGT_CONT cargado exitosamente: 2,052 registros

[PASO 2] Calculando campos monetarios...
✓ Monetarios agregados: 68,770 registros

[PASO 3] Creando tabla final con estructura Power BI...
✓ Tabla final estructurada: 68,770 registros

[PASO 4] Eliminando duplicados por clave primaria...
✓ Registros antes: 68,770
✓ Registros después: 45,970
✓ Duplicados eliminados: 22,800

[PASO 5] Optimizando y guardando tabla final...
✓ Tabla anterior eliminada

✅ TABLA FINAL GUARDADA:
  • Nombre: tbl_caratula_cdv_clean
  • Esquema: LH_INTELLIGENT_VISIBILITY_CARATULAS
  • Registros finales: 45,970
  • Particiones: 200

VERIFICACIÓN FINAL

📊 SAMPLE CR-LAVA PWOH 1 - 2025:
+---------+------+--------+--------+------+---------+---------+---+------+-------+-----------+------+-----------+
|Attribute|ORDENM|Inv_Inic|Llegadas|S&PO  |Facturado|Inv_Final|WHO|TTOS  |UND_TTO|Inv_Fina